In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings, os, time, gc
warnings.filterwarnings('ignore')

print("✅ All libraries loaded")
print("=" * 70)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.4 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total
✅ All libraries loaded


In [2]:
###############################################################################
# C0: COMPLETE SETUP — Option A Bottom-Up Analysis
# GSE182159 (243K cells, 59 subclusters, 23 donors, 5 groups)
# 2026-03-01
# 바닥부터: proportion → pathway → gene → pattern → theory
###############################################################################


# =================================================================
# CELL 1: Install & Import
# =================================================================

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.sparse import issparse
import os, warnings
warnings.filterwarnings('ignore')

print(f"scanpy:  {sc.__version__}")
print(f"anndata: {ad.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")
print("✅ Cell 1 complete")

scanpy:  1.12
anndata: 0.12.10
pandas:  2.2.2
numpy:   2.0.2
✅ Cell 1 complete


In [3]:
# =================================================================
# CELL 2: Google Drive Mount + Output Directory
# =================================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [44]:
"""
==========================================================================
V18 C3: INDIVIDUAL GENE EXPRESSION ANALYSIS
==========================================================================
169 genes × 6 lineages × Liver/Blood (tissue-separated)
Donor-level aggregation + Mann-Whitney U tests
GPU-accelerated (A100) with CuPy sparse matrix operations

DESIGN PRINCIPLES:
  - Bottom-up only: data shows what it shows
  - No combined Liver+Blood analysis
  - All comparisons include NL baseline
  - Dot/box plots only (no line graphs)
  - Liver=red circle, Blood=blue triangle
  - P-values always displayed

OUTPUT: /content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression/

"""

'\n==========================================================================\nV18 C3: INDIVIDUAL GENE EXPRESSION ANALYSIS\n==========================================================================\n169 genes × 6 lineages × Liver/Blood (tissue-separated)\nDonor-level aggregation + Mann-Whitney U tests\nGPU-accelerated (A100) with CuPy sparse matrix operations\n\nDESIGN PRINCIPLES:\n  - Bottom-up only: data shows what it shows\n  - No combined Liver+Blood analysis\n  - All comparisons include NL baseline\n  - Dot/box plots only (no line graphs)\n  - Liver=red circle, Blood=blue triangle\n  - P-values always displayed\n\nOUTPUT: /content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression/\n\n'

In [5]:
# ============================================================
# CELL 1: DATA LOADING
# ============================================================
print("\n" + "=" * 70)
print("  CELL 1: DATA LOADING")
print("=" * 70)

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression'
os.makedirs(RESULTS_DIR, exist_ok=True)

t0 = time.time()
adata = sc.read_h5ad(DATA_PATH)
print(f"✅ Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes in {time.time()-t0:.1f}s")

# --- Auto-detect column names (GSE182159 실제 컬럼에 완벽 대응) ---
print("  Available columns:", list(adata.obs.columns))

# Tissue column
tissue_col = None
for c in ['tissue', 'Tissue', 'tissue_type']:
    if c in adata.obs.columns:
        tissue_col = c
        break
assert tissue_col is not None

# Stage column
stage_col = None
for c in ['Stage', 'stage', 'disease_stage', 'disease_group', 'group', 'condition']:
    if c in adata.obs.columns:
        stage_col = c
        break
assert stage_col is not None

# Donor column ← 여기서 'sample'이 실제 donor ID임을 명확히 우선
donor_col = None
for c in ['sample', 'donor', 'sample_id', 'donor_id', 'orig.ident', 'patient', 'subject', 'Patient', 'Donor']:
    if c in adata.obs.columns:
        donor_col = c
        break
assert donor_col is not None, f"❌ No donor column found! Available: {list(adata.obs.columns)}"

# Lineage column ← 'major_lineage' 추가 (오류 원인 해결)
lineage_col = None
for c in ['major_lineage', 'lineage', 'Lineage', 'cell_lineage', 'celltype_major', 'gut2021_subcluster_v2']:
    if c in adata.obs.columns:
        lineage_col = c
        break
assert lineage_col is not None, f"❌ No lineage column found! Available: {list(adata.obs.columns)}"

print(f"✅ 자동 탐지 성공 → tissue={tissue_col}, stage={stage_col}, donor={donor_col}, lineage={lineage_col}")
print(f"  Tissues: {adata.obs[tissue_col].value_counts().to_dict()}")
print(f"  Stages: {adata.obs[stage_col].value_counts().to_dict()}")
print(f"  Lineages: {sorted(adata.obs[lineage_col].unique())}")
print(f"  Donors: {adata.obs[donor_col].nunique()} unique")



  CELL 1: DATA LOADING
✅ Loaded: 243,000 cells × 24,452 genes in 419.1s
  Available columns: ['sample', 'tissue', 'Stage', 'IT_cluster_21', 'IT_cluster_23', 'IT_cluster_25', 'IT_nk_collapse', 'IT_IT_signature', 'GSM_ID', 'IT_score_v2', 'IT_score_v3', 'IT_score_v4', 'IT_signature_final', 'IT_like', 'PW_mTOR_signaling', 'PW_glycolysis', 'PW_oxidative_phosphorylation', 'PW_nk_cell_cytotoxicity', 'PW_il15_signaling', 'PW_b_cell_differentiation', 'leiden', 'gut2021_subcluster', 'major_lineage', 'gut2021_subcluster_v2', 'TCR_clone.id', 'TCR_v_gene.x', 'TCR_j_gene.x', 'TCR_cdr3_nt.x', 'TCR_CType', 'BCR_clone.id', 'BCR_v_gene', 'BCR_j_gene', 'BCR_cdr3_nt', 'BCR_CType']
✅ 자동 탐지 성공 → tissue=tissue, stage=Stage, donor=sample, lineage=major_lineage
  Tissues: {'Blood': 136408, 'Liver': 106592}
  Stages: {'IA': 62545, 'IT': 49179, 'AR': 45452, 'CR': 43245, 'NL': 42579}
  Lineages: ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']
  Donors: 46 unique


In [6]:
# ============================================================
# CELL 2: DEFINE 169 GENES
# ============================================================
print("\n" + "=" * 70)
print("  CELL 2: DEFINING 169 TARGET GENES")
print("=" * 70)

# C3 original 152 genes (from v17 analysis)
C3_GENES = [
    # Inflammasome & innate sensing
    'AIM2', 'NLRC4', 'MEFV', 'CASP1', 'NLRP3', 'IL1B', 'GSDMD', 'PYCARD',
    'CASP4', 'CASP5', 'NAIP',
    'TLR2', 'TLR4', 'TLR7', 'TLR8', 'TLR9', 'MYD88', 'TICAM1',
    'DDX58', 'IFIH1', 'MAVS', 'CGAS', 'STING1',
    # Type I IFN
    'IFNA1', 'IFNB1', 'IFNAR1', 'IFNAR2', 'STAT1', 'STAT2', 'IRF3', 'IRF7',
    'IRF9', 'MX1', 'MX2', 'OAS1', 'ISG15', 'IFIT1',
    # Tolerance / suppression
    'LGALS9', 'TGFB1', 'IL10', 'IDO1', 'TNFAIP3', 'HAVCR2', 'CD274',
    'PDCD1LG2', 'VSIR', 'SIGLEC10', 'LILRB1', 'LILRB2',
    # Treg
    'FOXP3', 'IL2RA', 'CTLA4', 'IKZF2', 'TNFRSF18', 'TIGIT',
    'ENTPD1', 'NT5E', 'LRRC32', 'BACH2', 'PRDM1', 'IL7R',
    # T cell exhaustion
    'PDCD1', 'LAG3', 'TOX', 'EOMES', 'BATF', 'NFATC1',
    'TBX21', 'TCF7', 'SLAMF6', 'CXCR5', 'GZMB',
    # Terminal/progenitor exhaustion
    'ENTPD1', 'HAVCR2', 'CX3CR1', 'PRF1', 'GNLY', 'FGFBP2',
    'XCL1', 'XCL2', 'SELL',
    # Cytotoxicity & NK
    'GZMA', 'GZMK', 'GZMH', 'GZMM', 'NKG7', 'KLRK1', 'KLRD1',
    'NCR1', 'NCR3', 'FCGR3A', 'CD160', 'KIR2DL4',
    # mTOR / metabolism
    'MTOR', 'RPTOR', 'RICTOR', 'RPS6KB1', 'EIF4EBP1', 'AKT1',
    'HIF1A', 'LDHA', 'PKM', 'SLC2A1', 'PFKFB3',
    'NDUFS1', 'COX5A', 'ATP5F1A', 'UQCRC1', 'SDHB',
    'CPT1A', 'ACADVL', 'HADHA', 'PPARGC1A',
    # JAK-STAT
    'JAK1', 'JAK2', 'JAK3', 'TYK2', 'STAT3', 'STAT4', 'STAT5A',
    'STAT5B', 'STAT6', 'SOCS1', 'SOCS3', 'CISH', 'PIAS1',
    # B cell
    'CD19', 'MS4A1', 'CD79A', 'CD79B', 'PAX5', 'BCL6',
    'IRF4', 'XBP1', 'SDC1', 'TNFRSF17', 'MZB1',
    # Apoptosis
    'BCL2', 'MCL1', 'BCL2L1', 'BIRC3', 'CFLAR',
    'BAX', 'BAK1', 'BID', 'BBC3', 'CASP3', 'CASP8', 'FAS', 'FASLG',
    # DNA damage / epigenetic
    'TP53', 'ATM', 'ATR', 'BRCA1', 'CHEK1', 'CHEK2',
    'DNMT1', 'DNMT3A', 'TET2', 'HDAC1', 'EZH2', 'KDM6A',
    # Chemotaxis
    'CCR7', 'CXCR3', 'CXCR4', 'CXCR6', 'CCR2', 'CCR5',
    'CCL3', 'CCL4', 'CCL5', 'CXCL10', 'CXCL13', 'CX3CR1',
    # Antigen presentation
    'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1',
    'HLA-DPB1', 'B2M', 'TAP1', 'TAP2', 'CIITA', 'CD74',
    # Cancer-associated
    'TERT', 'MYC', 'VEGFA', 'HGF', 'MET', 'CTNNB1',
    'APC', 'AXIN1', 'TP53', 'RB1', 'CDKN2A', 'MDM2',
]

# C3b supplementary 17 genes
C3B_GENES = [
    'AICDA', 'JCHAIN', 'IL1RN', 'CD27', 'TOX2',
    'RICTOR',  # already in C3 but ensure presence
    'AIM2',    # already in C3 - B cell context
    'MEFV',    # already in C3
    'NLRC4',   # already in C3
    'CASP1',   # already in C3
    'LGALS9',  # already in C3
    'TGFB1',   # already in C3
    'MTOR',    # already in C3
    'JAK1',    # already in C3
    'RPTOR',   # already in C3
    'PYCARD',  # already in C3
    'PRDM1',   # already in C3
]

# Deduplicate and create final list
ALL_GENES = sorted(set(C3_GENES + C3B_GENES))
print(f"Total unique target genes: {len(ALL_GENES)}")

# Check which genes exist in the dataset
available_genes = [g for g in ALL_GENES if g in adata.var_names]
missing_genes = [g for g in ALL_GENES if g not in adata.var_names]
print(f"✅ Available in dataset: {len(available_genes)}")
if missing_genes:
    print(f"⚠️ Missing ({len(missing_genes)}): {missing_genes}")

# Save gene list
gene_df = pd.DataFrame({
    'gene': available_genes,
    'in_C3': [g in C3_GENES for g in available_genes],
    'in_C3b': [g in C3B_GENES for g in available_genes],
    'source': ['C3+C3b' if (g in C3_GENES and g in C3B_GENES) else 'C3' if g in C3_GENES else 'C3b'
               for g in available_genes]
})
gene_df.to_csv(f'{RESULTS_DIR}/C3_gene_list_{len(available_genes)}genes.csv', index=False)
print(f"✅ Gene list saved")



  CELL 2: DEFINING 169 TARGET GENES
Total unique target genes: 199
✅ Available in dataset: 196
⚠️ Missing (3): ['IFNA1', 'RPS6KB1', 'STING1']
✅ Gene list saved


In [7]:
# ============================================================
# CELL 3: GPU-ACCELERATED GENE EXPRESSION EXTRACTION
# ============================================================
print("\n" + "=" * 70)
print("  CELL 3: GPU-ACCELERATED GENE EXPRESSION EXTRACTION")
print("=" * 70)

# Define analysis groups
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
# Map potential alternative names
lineage_map = {}
for lin in adata.obs[lineage_col].unique():
    if 'CD4' in lin: lineage_map[lin] = 'CD4_T'
    elif 'CD8' in lin: lineage_map[lin] = 'CD8_T'
    elif lin.lower() in ['myeloid', 'nk', 'b', 'plasmab', 'gdt']:
        lineage_map[lin] = lin
    else:
        lineage_map[lin] = lin

STAGES = ['NL', 'IT', 'IA', 'AR', 'CR']
TISSUES = ['Liver', 'Blood']

def extract_gene_expression_gpu(adata, genes, tissue_val, batch_size=20000):
    """
    GPU-accelerated gene expression extraction.
    Uses CuPy for sparse matrix slicing and mean computation.
    Returns donor-level mean expression for each gene × lineage.
    """
    # Filter by tissue
    tissue_mask = adata.obs[tissue_col] == tissue_val
    adata_tissue = adata[tissue_mask]
    n_cells = adata_tissue.shape[0]
    print(f"\n  [{tissue_val}] {n_cells:,} cells")

    # Get gene indices
    gene_indices = []
    valid_genes = []
    for g in genes:
        if g in adata_tissue.var_names:
            gene_indices.append(list(adata_tissue.var_names).index(g))
            valid_genes.append(g)

    print(f"  Valid genes: {len(valid_genes)}/{len(genes)}")

    # Extract expression matrix (cells × genes) — sparse
    t0 = time.time()
    X = adata_tissue.X  # sparse matrix

    # Convert to CSC for efficient column slicing
    if sp.issparse(X):
        X_csc = X.tocsc()
    else:
        X_csc = sp.csc_matrix(X)

    # Slice only target genes → dense array (cells × n_genes)
    # Process in batches to avoid memory overflow
    n_genes = len(gene_indices)
    print(f"  Extracting {n_genes} genes from {n_cells:,} cells...")

    # GPU-accelerated: transfer gene columns in batches
    expr_matrix = np.zeros((n_cells, n_genes), dtype=np.float32)

    GENE_BATCH = 50  # Process 50 genes at a time
    for i in range(0, n_genes, GENE_BATCH):
        batch_idx = gene_indices[i:i+GENE_BATCH]
        batch_data = X_csc[:, batch_idx].toarray().astype(np.float32)
        expr_matrix[:, i:i+len(batch_idx)] = batch_data

    print(f"  ✅ Expression extracted in {time.time()-t0:.1f}s")

    # GPU-accelerated donor-level aggregation
    t1 = time.time()
    obs = adata_tissue.obs
    results = []

    for lineage in LINEAGES:
        # Get lineage mask
        lin_mask = obs[lineage_col].map(lineage_map).values == lineage
        if lin_mask.sum() == 0:
            continue

        lin_expr = expr_matrix[lin_mask]  # (cells_in_lineage × n_genes)
        lin_obs = obs[lin_mask]

        # Transfer to GPU for fast aggregation
        lin_expr_gpu = cp.asarray(lin_expr)

        for donor_id in lin_obs[donor_col].unique():
            donor_mask = (lin_obs[donor_col].values == donor_id)
            n_donor_cells = donor_mask.sum()
            if n_donor_cells == 0:
                continue

            stage = lin_obs[lin_obs[donor_col] == donor_id][stage_col].iloc[0]

            # GPU mean computation
            donor_expr_gpu = lin_expr_gpu[cp.asarray(donor_mask)]
            donor_means = cp.asnumpy(cp.mean(donor_expr_gpu, axis=0))

            row = {
                'tissue': tissue_val,
                'lineage': lineage,
                'donor': donor_id,
                'Stage': stage,
                'n_cells': int(n_donor_cells),
            }
            for j, gene in enumerate(valid_genes):
                row[gene] = float(donor_means[j])

            results.append(row)

        # Free GPU memory after each lineage
        del lin_expr_gpu
        cp.get_default_memory_pool().free_all_blocks()

    print(f"  ✅ Donor aggregation in {time.time()-t1:.1f}s ({len(results)} rows)")
    return pd.DataFrame(results), valid_genes


# Run extraction
print("\n" + "=" * 50)
print("  EXTRACTING: LIVER")
print("=" * 50)
liver_df, valid_genes = extract_gene_expression_gpu(adata, available_genes, 'Liver')

print("\n" + "=" * 50)
print("  EXTRACTING: BLOOD")
print("=" * 50)
blood_df, _ = extract_gene_expression_gpu(adata, available_genes, 'Blood')

# Save raw donor-level data
liver_df.to_csv(f'{RESULTS_DIR}/C3_liver_donor_gene_expression.csv.gz',
                index=False, compression='gzip')
blood_df.to_csv(f'{RESULTS_DIR}/C3_blood_donor_gene_expression.csv.gz',
                index=False, compression='gzip')
print(f"\n✅ Saved: Liver ({liver_df.shape}), Blood ({blood_df.shape})")
print(f"✅ Valid genes for analysis: {len(valid_genes)}")


  CELL 3: GPU-ACCELERATED GENE EXPRESSION EXTRACTION

  EXTRACTING: LIVER

  [Liver] 106,592 cells
  Valid genes: 196/196
  Extracting 196 genes from 106,592 cells...
  ✅ Expression extracted in 27.2s
  ✅ Donor aggregation in 1.4s (136 rows)

  EXTRACTING: BLOOD

  [Blood] 136,408 cells
  Valid genes: 196/196
  Extracting 196 genes from 136,408 cells...
  ✅ Expression extracted in 33.9s
  ✅ Donor aggregation in 0.3s (138 rows)

✅ Saved: Liver ((136, 201)), Blood ((138, 201))
✅ Valid genes for analysis: 196


In [8]:
# ============================================================
# CELL 4: STATISTICAL TESTING — Mann-Whitney U (All comparisons)
# ============================================================
print("\n" + "=" * 70)
print("  CELL 4: STATISTICAL TESTING")
print("=" * 70)

def run_statistics(df, tissue_name, genes, comparisons=None):
    """
    Run Mann-Whitney U tests for all gene × lineage × comparison combinations.
    Default comparisons: NL vs IT, NL vs IA, NL vs AR, NL vs CR, IT vs IA, IA vs AR, CR vs AR
    """
    if comparisons is None:
        comparisons = [
            ('NL', 'IT'), ('NL', 'IA'), ('NL', 'AR'), ('NL', 'CR'),
            ('IT', 'IA'), ('IA', 'AR'), ('CR', 'AR'),
        ]

    results = []
    total = len(LINEAGES) * len(genes) * len(comparisons)
    done = 0

    for lineage in LINEAGES:
        lin_df = df[df['lineage'] == lineage]
        if len(lin_df) == 0:
            continue

        for gene in genes:
            if gene not in lin_df.columns:
                continue

            for grp1, grp2 in comparisons:
                vals1 = lin_df[lin_df['Stage'] == grp1][gene].dropna().values
                vals2 = lin_df[lin_df['Stage'] == grp2][gene].dropna().values

                n1, n2 = len(vals1), len(vals2)
                if n1 < 2 or n2 < 2:
                    continue

                mean1, mean2 = np.mean(vals1), np.mean(vals2)

                # Percentage change
                if mean1 > 0:
                    pct_change = ((mean2 - mean1) / mean1) * 100
                elif mean2 > 0:
                    pct_change = 999.0  # 0→positive
                else:
                    pct_change = 0.0

                direction = '↑' if mean2 > mean1 else '↓' if mean2 < mean1 else '='

                # Mann-Whitney U test
                try:
                    U, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
                except:
                    p = 1.0
                    U = 0

                # Consistency: count donor pairs where direction holds
                n_pairs = n1 * n2
                concordant = 0
                for v1 in vals1:
                    for v2 in vals2:
                        if direction == '↑' and v2 > v1:
                            concordant += 1
                        elif direction == '↓' and v2 < v1:
                            concordant += 1
                        elif direction == '=' and v2 == v1:
                            concordant += 1
                consistency = f"{concordant}/{n_pairs}"

                # Significance annotation
                if p < 0.01:
                    sig = '**'
                elif p < 0.05:
                    sig = '*'
                elif p < 0.10:
                    sig = '†'
                else:
                    sig = 'NS'

                results.append({
                    'tissue': tissue_name,
                    'lineage': lineage,
                    'gene': gene,
                    'comparison': f'{grp1}→{grp2}',
                    'grp1': grp1,
                    'grp2': grp2,
                    'n1': n1,
                    'n2': n2,
                    'mean_grp1': round(mean1, 6),
                    'mean_grp2': round(mean2, 6),
                    'pct_change': round(pct_change, 1),
                    'direction': direction,
                    'consistency': consistency,
                    'U_stat': U,
                    'p_value': round(p, 6),
                    'sig': sig,
                })

            done += len(comparisons)
            if done % 500 == 0:
                print(f"  [{tissue_name}] {done}/{total} tests ({done*100//total}%)")

    return pd.DataFrame(results)


t0 = time.time()

print(f"\n--- Liver Statistics ---")
liver_stats = run_statistics(liver_df, 'Liver', valid_genes)

print(f"\n--- Blood Statistics ---")
blood_stats = run_statistics(blood_df, 'Blood', valid_genes)

# Combine
all_stats = pd.concat([liver_stats, blood_stats], ignore_index=True)

# Save
all_stats.to_csv(f'{RESULTS_DIR}/C3_all_statistics.csv.gz', index=False, compression='gzip')

# Summary
n_sig_liver = (liver_stats['p_value'] < 0.05).sum()
n_sig_blood = (blood_stats['p_value'] < 0.05).sum()
n_trend_liver = ((liver_stats['p_value'] >= 0.05) & (liver_stats['p_value'] < 0.10)).sum()
n_trend_blood = ((blood_stats['p_value'] >= 0.05) & (blood_stats['p_value'] < 0.10)).sum()

print(f"\n✅ Statistics complete in {time.time()-t0:.1f}s")
print(f"  Liver: {n_sig_liver} significant (p<0.05), {n_trend_liver} trend (p<0.10), total={len(liver_stats)}")
print(f"  Blood: {n_sig_blood} significant (p<0.05), {n_trend_blood} trend (p<0.10), total={len(blood_stats)}")



  CELL 4: STATISTICAL TESTING

--- Liver Statistics ---
  [Liver] 3500/8232 tests (42%)
  [Liver] 7000/8232 tests (85%)

--- Blood Statistics ---
  [Blood] 3500/8232 tests (42%)
  [Blood] 7000/8232 tests (85%)

✅ Statistics complete in 28.1s
  Liver: 583 significant (p<0.05), 434 trend (p<0.10), total=8232
  Blood: 859 significant (p<0.05), 585 trend (p<0.10), total=8232


In [9]:
# ============================================================
# CELL 5: TOP FINDINGS — NL→IT (IT-specific candidates)
# ============================================================
print("\n" + "=" * 70)
print("  CELL 5: TOP NL→IT FINDINGS (IT-SPECIFIC CANDIDATES)")
print("=" * 70)

it_stats = all_stats[all_stats['comparison'] == 'NL→IT'].copy()
it_sig = it_stats[it_stats['p_value'] < 0.05].sort_values('p_value')

print(f"\nSignificant NL→IT changes (p<0.05): {len(it_sig)}")
print(f"\n{'Tissue':<8} {'Lineage':<10} {'Gene':<12} {'Change':>10} {'Dir':>4} {'Consistency':>12} {'p-value':>10}")
print("-" * 70)
for _, row in it_sig.head(40).iterrows():
    print(f"{row['tissue']:<8} {row['lineage']:<10} {row['gene']:<12} "
          f"{row['pct_change']:>8.1f}% {row['direction']:>4} "
          f"{row['consistency']:>12} {row['p_value']:>10.4f}{row['sig']}")

# IT trends
it_trend = it_stats[(it_stats['p_value'] >= 0.05) & (it_stats['p_value'] < 0.10)].sort_values('p_value')
print(f"\n\nNL→IT Trends (0.05 ≤ p < 0.10): {len(it_trend)}")
for _, row in it_trend.head(20).iterrows():
    print(f"{row['tissue']:<8} {row['lineage']:<10} {row['gene']:<12} "
          f"{row['pct_change']:>8.1f}% {row['direction']:>4} "
          f"{row['consistency']:>12} {row['p_value']:>10.4f}{row['sig']}")

# Save IT-specific results
it_stats.to_csv(f'{RESULTS_DIR}/C3_NL_vs_IT_all.csv', index=False)
it_sig.to_csv(f'{RESULTS_DIR}/C3_NL_vs_IT_significant.csv', index=False)



  CELL 5: TOP NL→IT FINDINGS (IT-SPECIFIC CANDIDATES)

Significant NL→IT changes (p<0.05): 392

Tissue   Lineage    Gene             Change  Dir  Consistency    p-value
----------------------------------------------------------------------
Liver    Myeloid    BAK1            379.5%    ↑        36/36     0.0022**
Liver    CD4_T      FGFBP2          366.8%    ↑        36/36     0.0022**
Liver    CD4_T      PRDM1           -36.4%    ↓        36/36     0.0022**
Liver    CD4_T      NCR3            -50.2%    ↓        36/36     0.0022**
Liver    CD4_T      XCL1            237.5%    ↑        36/36     0.0022**
Liver    CD8_T      APC              84.2%    ↑        36/36     0.0022**
Liver    CD8_T      DDX58           110.2%    ↑        36/36     0.0022**
Liver    CD8_T      TOX              86.6%    ↑        36/36     0.0022**
Liver    CD8_T      XCL1            253.4%    ↑        36/36     0.0022**
Liver    Myeloid    CD74             34.1%    ↑        36/36     0.0022**
Liver    Myeloid   

In [10]:
# ============================================================
# CELL 6: CROSS-TISSUE DISCREPANCY ANALYSIS
# ============================================================
print("\n" + "=" * 70)
print("  CELL 6: LIVER vs BLOOD DISCREPANCY (NL→IT)")
print("=" * 70)

# Pivot: for each gene × lineage, compare Liver vs Blood NL→IT
liver_it = it_stats[it_stats['tissue'] == 'Liver'][['lineage', 'gene', 'pct_change', 'p_value', 'sig']].copy()
liver_it.columns = ['lineage', 'gene', 'liver_pct', 'liver_p', 'liver_sig']

blood_it = it_stats[it_stats['tissue'] == 'Blood'][['lineage', 'gene', 'pct_change', 'p_value', 'sig']].copy()
blood_it.columns = ['lineage', 'gene', 'blood_pct', 'blood_p', 'blood_sig']

cross = liver_it.merge(blood_it, on=['lineage', 'gene'], how='outer')
cross['discrepancy'] = abs(cross['liver_pct'].fillna(0) - cross['blood_pct'].fillna(0))
cross['direction_match'] = (
    (cross['liver_pct'].fillna(0) > 0) == (cross['blood_pct'].fillna(0) > 0)
)

# Flag high discrepancies
cross = cross.sort_values('discrepancy', ascending=False)
cross.to_csv(f'{RESULTS_DIR}/C3_liver_blood_discrepancy_NL_IT.csv', index=False)

print(f"\n--- Top 20 Liver vs Blood Discrepancies (NL→IT) ---")
print(f"{'Lineage':<10} {'Gene':<12} {'Liver%':>10} {'Blood%':>10} {'Disc':>8} {'Dir Match':>10}")
print("-" * 65)
for _, row in cross.head(20).iterrows():
    lp = f"{row['liver_pct']:.1f}%{row['liver_sig']}" if pd.notna(row['liver_pct']) else "N/A"
    bp = f"{row['blood_pct']:.1f}%{row['blood_sig']}" if pd.notna(row['blood_pct']) else "N/A"
    match = "✅" if row['direction_match'] else "⚠️"
    print(f"{row['lineage']:<10} {row['gene']:<12} {lp:>10} {bp:>10} {row['discrepancy']:>8.1f} {match:>10}")




  CELL 6: LIVER vs BLOOD DISCREPANCY (NL→IT)

--- Top 20 Liver vs Blood Discrepancies (NL→IT) ---
Lineage    Gene             Liver%     Blood%     Disc  Dir Match
-----------------------------------------------------------------
CD4_T      IL1B           8055.8%*   -96.5%NS   8152.3         ⚠️
PlasmaB    CXCR5         7906.9%NS   -14.2%NS   7921.1         ⚠️
PlasmaB    TLR7          9146.5%**  1578.2%**   7568.3          ✅
CD8_T      CXCL13         7404.9%*  -100.0%NS   7504.9         ⚠️
CD4_T      CD19          6112.2%NS   161.7%NS   5950.5          ✅
PlasmaB    CIITA           420.3%*  5967.0%**   5546.7          ✅
PlasmaB    TCF7          5192.7%NS   104.3%NS   5088.4          ✅
PlasmaB    MS4A1          4340.7%*   149.9%NS   4190.8          ✅
B          AICDA           999.0%*   4834.1%*   3835.1          ✅
PlasmaB    XCL1          2390.8%NS     0.0%NS   2390.8         ⚠️
PlasmaB    ATR              50.4%†   2099.3%*   2048.9          ✅
Myeloid    TOX           1856.8%**   -91.1%

In [11]:
# ============================================================
# CELL 7: PATTERN CLASSIFICATION (All comparisons)
# ============================================================
print("\n" + "=" * 70)
print("  CELL 7: PATTERN CLASSIFICATION")
print("=" * 70)

def classify_pattern(gene_lineage_stats):
    """
    Classify gene-lineage combination into disease spectrum patterns.
    Based on NL→X significance patterns:
      - IT-specific: NL→IT sig, NL→IA NS
      - Chronic_persistent: NL→IT sig AND NL→IA sig (same direction)
      - IA-emergent: NL→IA sig, NL→IT NS
      - AR-advantage: NL→AR sig, NL→IT NS, NL→IA NS
      - CR-scar: NL→CR sig (unique to CR)
    """
    nl_it = gene_lineage_stats.get('NL→IT', {})
    nl_ia = gene_lineage_stats.get('NL→IA', {})
    nl_ar = gene_lineage_stats.get('NL→AR', {})
    nl_cr = gene_lineage_stats.get('NL→CR', {})

    it_sig = nl_it.get('p', 1) < 0.05
    ia_sig = nl_ia.get('p', 1) < 0.05
    ar_sig = nl_ar.get('p', 1) < 0.05
    cr_sig = nl_cr.get('p', 1) < 0.05

    it_trend = nl_it.get('p', 1) < 0.10
    ia_trend = nl_ia.get('p', 1) < 0.10

    if it_sig and not ia_sig:
        return 'IT-specific'
    elif it_sig and ia_sig:
        # Check if same direction
        it_dir = nl_it.get('dir', '=')
        ia_dir = nl_ia.get('dir', '=')
        if it_dir == ia_dir:
            return 'Chronic_persistent'
        else:
            return 'IT-specific (reversed in IA)'
    elif not it_sig and ia_sig:
        return 'IA-emergent'
    elif not it_sig and not ia_sig and ar_sig:
        return 'AR-advantage'
    elif not it_sig and not ia_sig and not ar_sig and cr_sig:
        return 'CR-scar'
    elif it_trend and not ia_trend:
        return 'IT-trend'
    else:
        return 'NS'


# Classify for each tissue
pattern_results = []

for tissue in TISSUES:
    tissue_stats = all_stats[all_stats['tissue'] == tissue]

    for lineage in LINEAGES:
        lin_stats = tissue_stats[tissue_stats['lineage'] == lineage]

        for gene in valid_genes:
            gene_stats = lin_stats[lin_stats['gene'] == gene]

            comp_dict = {}
            for _, row in gene_stats.iterrows():
                comp_dict[row['comparison']] = {
                    'p': row['p_value'],
                    'pct': row['pct_change'],
                    'dir': row['direction'],
                    'sig': row['sig'],
                    'consistency': row['consistency'],
                }

            pattern = classify_pattern(comp_dict)

            nl_it = comp_dict.get('NL→IT', {})
            nl_ia = comp_dict.get('NL→IA', {})
            nl_cr = comp_dict.get('NL→CR', {})

            pattern_results.append({
                'tissue': tissue,
                'lineage': lineage,
                'gene': gene,
                'pattern': pattern,
                'NL_IT_pct': nl_it.get('pct', None),
                'NL_IT_p': nl_it.get('p', None),
                'NL_IT_sig': nl_it.get('sig', 'N/A'),
                'NL_IA_pct': nl_ia.get('pct', None),
                'NL_IA_p': nl_ia.get('p', None),
                'NL_IA_sig': nl_ia.get('sig', 'N/A'),
                'NL_CR_pct': nl_cr.get('pct', None),
                'NL_CR_p': nl_cr.get('p', None),
                'NL_CR_sig': nl_cr.get('sig', 'N/A'),
            })

pattern_df = pd.DataFrame(pattern_results)
pattern_df.to_csv(f'{RESULTS_DIR}/C3_pattern_classification.csv', index=False)

# Summary
print("\n--- Pattern Distribution ---")
for tissue in TISSUES:
    t_df = pattern_df[pattern_df['tissue'] == tissue]
    counts = t_df['pattern'].value_counts()
    print(f"\n  [{tissue}]")
    for pat, cnt in counts.items():
        if pat != 'NS':
            print(f"    {pat:<30} {cnt:>5}")

# IT-specific genes
it_specific = pattern_df[pattern_df['pattern'] == 'IT-specific']
print(f"\n\n--- IT-SPECIFIC Genes (tissue-separated) ---")
for tissue in TISSUES:
    t_df = it_specific[it_specific['tissue'] == tissue].sort_values('NL_IT_p')
    print(f"\n  [{tissue}] ({len(t_df)} gene-lineage combinations)")
    for _, row in t_df.head(20).iterrows():
        print(f"    {row['lineage']:<10} {row['gene']:<12} {row['NL_IT_pct']:>8.1f}% "
              f"p={row['NL_IT_p']:.4f}{row['NL_IT_sig']}")



  CELL 7: PATTERN CLASSIFICATION

--- Pattern Distribution ---

  [Liver]
    IT-specific                       85
    IA-emergent                       73
    Chronic_persistent                54
    AR-advantage                      48
    CR-scar                           46
    IT-trend                          41

  [Blood]
    IT-specific                      169
    Chronic_persistent                84
    CR-scar                           72
    IA-emergent                       41
    IT-trend                          27
    AR-advantage                      25


--- IT-SPECIFIC Genes (tissue-separated) ---

  [Liver] (85 gene-lineage combinations)
    Myeloid    HLA-DRA          41.9% p=0.0022**
    Myeloid    HLA-DRB1         48.6% p=0.0022**
    Myeloid    HLA-DPB1         94.8% p=0.0022**
    Myeloid    DNMT1           125.2% p=0.0022**
    CD4_T      FGFBP2          366.8% p=0.0022**
    CD8_T      TOX              86.6% p=0.0022**
    CD8_T      XCL1            253.4% p

In [12]:
# ============================================================
# CELL 8: VISUALIZATION — Dot/Box Plots for Top Findings
# ============================================================
print("\n" + "=" * 70)
print("  CELL 8: GENERATING DOT/BOX PLOTS")
print("=" * 70)

def plot_gene_dotbox(liver_df, blood_df, gene, lineage, save_dir,
                     liver_stats_df=None, blood_stats_df=None):
    """
    Create side-by-side dot/box plot for a gene × lineage.
    Left: Liver (red circles), Right: Blood (blue triangles)
    All 5 groups shown, NL always included.
    P-values displayed.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    stage_order = ['NL', 'IT', 'IA', 'AR', 'CR']
    stage_colors = {'NL': '#2ecc71', 'IT': '#e74c3c', 'IA': '#e67e22',
                    'AR': '#3498db', 'CR': '#9b59b6'}

    for ax_idx, (df, tissue, color, marker, label) in enumerate([
        (liver_df, 'Liver', '#e74c3c', 'o', 'Liver'),
        (blood_df, 'Blood', '#3498db', '^', 'Blood')
    ]):
        ax = axes[ax_idx]
        lin_df = df[df['lineage'] == lineage]

        if gene not in lin_df.columns or len(lin_df) == 0:
            ax.set_title(f'{tissue}: No data', fontsize=12)
            continue

        positions = []
        box_data = []

        for i, stage in enumerate(stage_order):
            vals = lin_df[lin_df['Stage'] == stage][gene].dropna().values
            if len(vals) > 0:
                # Jittered dots
                jitter = np.random.uniform(-0.15, 0.15, len(vals))
                ax.scatter([i] * len(vals) + jitter, vals,
                          c=stage_colors[stage], marker=marker, s=80,
                          edgecolors='black', linewidths=0.5, alpha=0.8,
                          zorder=3)
                positions.append(i)
                box_data.append(vals)

                # Mean line
                ax.hlines(np.mean(vals), i - 0.3, i + 0.3,
                         colors='black', linewidths=2, zorder=4)

        # Box plot (transparent)
        if box_data:
            bp = ax.boxplot(box_data, positions=positions, widths=0.5,
                           patch_artist=True, showfliers=False, zorder=2)
            for patch in bp['boxes']:
                patch.set_facecolor('white')
                patch.set_alpha(0.3)

        # P-values for NL comparisons
        stats_df = liver_stats_df if tissue == 'Liver' else blood_stats_df
        if stats_df is not None:
            gene_lin_stats = stats_df[
                (stats_df['gene'] == gene) & (stats_df['lineage'] == lineage)
            ]
            for _, row in gene_lin_stats.iterrows():
                if row['grp1'] == 'NL' and row['p_value'] < 0.10:
                    grp2_idx = stage_order.index(row['grp2'])
                    p_text = f"p={row['p_value']:.3f}{row['sig']}"
                    y_max = ax.get_ylim()[1]
                    ax.annotate(p_text, xy=(grp2_idx, y_max * 0.95),
                              fontsize=8, ha='center', color='red' if row['p_value'] < 0.05 else 'gray')

        ax.set_xticks(range(len(stage_order)))
        ax.set_xticklabels(stage_order, fontsize=11)
        ax.set_title(f'{tissue}', fontsize=14, fontweight='bold', color=color)
        ax.set_ylabel('Expression' if ax_idx == 0 else '', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.suptitle(f'{gene} — {lineage}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    fname = f'{save_dir}/fig_C3_{gene}_{lineage}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return fname


# Create figures directory
fig_dir = f'{RESULTS_DIR}/figures'
os.makedirs(fig_dir, exist_ok=True)

# Generate plots for all significant findings
sig_genes = set()
for _, row in all_stats[all_stats['p_value'] < 0.05].iterrows():
    sig_genes.add((row['gene'], row['lineage']))

# Also add key v17 genes of interest
key_genes = [
    ('AIM2', 'Myeloid'), ('NLRC4', 'Myeloid'), ('MEFV', 'Myeloid'),
    ('CASP1', 'Myeloid'), ('LGALS9', 'Myeloid'), ('TGFB1', 'Myeloid'),
    ('AICDA', 'B'), ('JCHAIN', 'B'), ('JCHAIN', 'PlasmaB'),
    ('MTOR', 'Myeloid'), ('MTOR', 'CD4_T'), ('MTOR', 'NK'),
    ('JAK1', 'Myeloid'), ('JAK1', 'CD4_T'),
    ('RICTOR', 'Myeloid'), ('RICTOR', 'CD4_T'),
    ('IL1RN', 'CD4_T'), ('IL1RN', 'CD8_T'),
    ('AIM2', 'B'), ('TOX2', 'Myeloid'), ('CD27', 'CD4_T'),
    ('PYCARD', 'Myeloid'), ('PYCARD', 'PlasmaB'),
]
sig_genes.update(key_genes)

print(f"Generating {len(sig_genes)} dot/box plots...")
t0 = time.time()

for i, (gene, lineage) in enumerate(sorted(sig_genes)):
    if gene in valid_genes:
        plot_gene_dotbox(liver_df, blood_df, gene, lineage, fig_dir,
                        liver_stats, blood_stats)
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{len(sig_genes)}] plots done")

print(f"✅ All plots generated in {time.time()-t0:.1f}s")


  CELL 8: GENERATING DOT/BOX PLOTS
Generating 609 dot/box plots...
  [20/609] plots done
  [40/609] plots done
  [60/609] plots done
  [80/609] plots done
  [100/609] plots done
  [120/609] plots done
  [140/609] plots done
  [160/609] plots done
  [180/609] plots done
  [200/609] plots done
  [220/609] plots done
  [240/609] plots done
  [260/609] plots done
  [280/609] plots done
  [300/609] plots done
  [320/609] plots done
  [340/609] plots done
  [360/609] plots done
  [380/609] plots done
  [400/609] plots done
  [420/609] plots done
  [440/609] plots done
  [460/609] plots done
  [480/609] plots done
  [500/609] plots done
  [520/609] plots done
  [540/609] plots done
  [560/609] plots done
  [580/609] plots done
  [600/609] plots done
✅ All plots generated in 251.8s


In [13]:
# ============================================================
# CELL 9: V17 COMPARISON — Armed Tolerance Genes (Liver-only)
# ============================================================
print("\n" + "=" * 70)
print("  CELL 9: v17 KEY FINDINGS REPLICATION CHECK (Liver-only)")
print("=" * 70)

armed_tolerance_genes = {
    'Sensors': ['AIM2', 'NLRC4', 'MEFV', 'CASP1'],
    'Suppressors': ['LGALS9', 'TGFB1'],
    'Pan-immune': ['MTOR', 'JAK1', 'RPTOR', 'PYCARD'],
    'B cell': ['AICDA', 'JCHAIN', 'PRDM1'],
    'CR scar': ['RICTOR'],
    'C3b new': ['IL1RN', 'CD27', 'TOX2', 'AIM2'],
}

print("\n--- v17 Key Findings: Liver-only Replication ---")
print(f"{'Category':<15} {'Gene':<10} {'Lineage':<10} {'v18 Liver%':>12} {'p-value':>10} {'Sig':>4} {'Consistency':>12}")
print("-" * 80)

liver_nl_it = liver_stats[liver_stats['comparison'] == 'NL→IT']

for category, genes in armed_tolerance_genes.items():
    for gene in genes:
        gene_data = liver_nl_it[liver_nl_it['gene'] == gene]
        for _, row in gene_data.iterrows():
            print(f"{category:<15} {gene:<10} {row['lineage']:<10} "
                  f"{row['pct_change']:>10.1f}% {row['p_value']:>10.4f} {row['sig']:>4} "
                  f"{row['consistency']:>12}")



  CELL 9: v17 KEY FINDINGS REPLICATION CHECK (Liver-only)

--- v17 Key Findings: Liver-only Replication ---
Category        Gene       Lineage      v18 Liver%    p-value  Sig  Consistency
--------------------------------------------------------------------------------
Sensors         AIM2       Myeloid         126.7%     0.1922   NS        25/36
Sensors         AIM2       CD4_T            39.1%     0.4680   NS        22/36
Sensors         AIM2       CD8_T             5.1%     1.0000   NS        18/36
Sensors         AIM2       NK                4.1%     1.0000   NS        18/36
Sensors         AIM2       B               114.7%     0.0519    †        26/30
Sensors         AIM2       PlasmaB         -24.4%     0.7823   NS        12/30
Sensors         NLRC4      Myeloid          38.1%     0.5725   NS        21/36
Sensors         NLRC4      CD4_T            37.3%     1.0000   NS         6/36
Sensors         NLRC4      CD8_T             0.0%     1.0000   NS        36/36
Sensors         NLR

In [14]:
# ============================================================
# CELL 10: SUMMARY & STATUS
# ============================================================
print("\n" + "=" * 70)
print("  V18 C3: COMPLETE SUMMARY")
print("=" * 70)

print(f"""
V18-C3: INDIVIDUAL GENE EXPRESSION (Tissue-Separated, GPU-accelerated)
================================================================
Genes analyzed: {len(valid_genes)}
Lineages: {', '.join(LINEAGES)}
Tissues: Liver, Blood (NO combined)
Comparisons: NL→IT, NL→IA, NL→AR, NL→CR, IT→IA, IA→AR, CR→AR

SIGNIFICANT (p<0.05):
  Liver: {n_sig_liver}
  Blood: {n_sig_blood}

PATTERN CLASSIFICATION:
""")

for tissue in TISSUES:
    t_df = pattern_df[pattern_df['tissue'] == tissue]
    counts = t_df['pattern'].value_counts()
    print(f"  [{tissue}]")
    for pat, cnt in counts.items():
        if pat != 'NS':
            print(f"    {pat}: {cnt}")

print(f"""
FILES: {RESULTS_DIR}/
  - C3_gene_list_{len(valid_genes)}genes.csv
  - C3_liver_donor_gene_expression.csv.gz
  - C3_blood_donor_gene_expression.csv.gz
  - C3_all_statistics.csv.gz
  - C3_NL_vs_IT_all.csv
  - C3_NL_vs_IT_significant.csv
  - C3_liver_blood_discrepancy_NL_IT.csv
  - C3_pattern_classification.csv
  - figures/ (dot/box plots for significant genes)

STATUS: ✅ C3 COMPLETE — Ready for C4
""")

print("=" * 70)
print("  v18-C3 COMPLETE")
print("=" * 70)



  V18 C3: COMPLETE SUMMARY

V18-C3: INDIVIDUAL GENE EXPRESSION (Tissue-Separated, GPU-accelerated)
Genes analyzed: 196
Lineages: Myeloid, CD4_T, CD8_T, NK, B, PlasmaB
Tissues: Liver, Blood (NO combined)
Comparisons: NL→IT, NL→IA, NL→AR, NL→CR, IT→IA, IA→AR, CR→AR

SIGNIFICANT (p<0.05):
  Liver: 583
  Blood: 859

PATTERN CLASSIFICATION:

  [Liver]
    IT-specific: 85
    IA-emergent: 73
    Chronic_persistent: 54
    AR-advantage: 48
    CR-scar: 46
    IT-trend: 41
  [Blood]
    IT-specific: 169
    Chronic_persistent: 84
    CR-scar: 72
    IA-emergent: 41
    IT-trend: 27
    AR-advantage: 25

FILES: /content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression/
  - C3_gene_list_196genes.csv
  - C3_liver_donor_gene_expression.csv.gz
  - C3_blood_donor_gene_expression.csv.gz
  - C3_all_statistics.csv.gz
  - C3_NL_vs_IT_all.csv
  - C3_NL_vs_IT_significant.csv
  - C3_liver_blood_discrepancy_NL_IT.csv
  - C3_pattern_classification.csv
  - figures/ (dot/box plots for signifi

In [45]:
"""
########################################################################
# V18-C4: 26 PATHWAY SCORING — Tissue-Separated, Liver & Blood 동시
# ============================================================
# Step (2) of 7-step pipeline
# Prereq: C3 completed (proportions)
# Input:  adata_liver, adata_blood, lineage_map (from C3)
# Output: version18-analysis/C4_pathway/
########################################################################

# %% [markdown]
# # V18-C4: 26 Pathway × 6 Lineage Scoring (Tissue-Separated)
# **Step 2/7**: Score 26 gene pathways per lineage, compare across disease spectrum, Liver AND Blood simultaneously.


OUTPUT: /content/drive/MyDrive/ITLAS/results/version18-analysis/C4_save/
==========================================================================
"""

'\n########################################################################\n# V18-C4: 26 PATHWAY SCORING — Tissue-Separated, Liver & Blood 동시\n# ============================================================\n# Step (2) of 7-step pipeline\n# Prereq: C3 completed (proportions)\n# Input:  adata_liver, adata_blood, lineage_map (from C3)\n# Output: version18-analysis/C4_pathway/\n########################################################################\n\n# %% [markdown]\n# # V18-C4: 26 Pathway × 6 Lineage Scoring (Tissue-Separated)\n# **Step 2/7**: Score 26 gene pathways per lineage, compare across disease spectrum, Liver AND Blood simultaneously.\n\n\nOUTPUT: /content/drive/MyDrive/ITLAS/results/version18-analysis/C4_save/\n==========================================================================\n'

In [16]:
# ============================================================
#  CELL 14: DEFINE 26 GENE SETS
# ============================================================
print("=" * 70)
print("  C4: 26 PATHWAY GENE SETS")
print("=" * 70)

C4_SAVE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C4_pathway/'
os.makedirs(C4_SAVE, exist_ok=True)

gene_sets = {
    'inflammasome':       ['NLRP3','AIM2','NLRC4','MEFV','PYCARD','CASP1'],
    'cytotoxicity':       ['GZMA','GZMB','GZMK','GNLY','PRF1','FASLG','NKG7'],
    'checkpoint':         ['PDCD1','LAG3','HAVCR2','TIGIT','CTLA4','CD160'],
    'exhaustion':         ['TOX','TOX2','ENTPD1','EOMES','BATF','LAYN'],
    'nk_function':        ['NCAM1','NCR1','NCR3','KLRK1','KLRD1','FCGR3A'],
    'nk_il15_dual':       ['GZMB','GNLY','PRF1','NKG7','KLRD1','FCGR3A','FCER1G','TYROBP'],
    'il15_mtor':          ['MTOR','RPTOR','RICTOR','EIF4EBP1','JAK1','JAK3','STAT5A','STAT5B','IL2RG','IL15RA'],
    'immune_evasion':     ['CD274','PDCD1LG2','LGALS9','VTCN1','HAVCR2','TGFB1','IDO1','TNFAIP3'],
    'treg':               ['FOXP3','IL2RA','CTLA4','TIGIT','IKZF2'],
    'naive_t':            ['TCF7','LEF1','SELL','CCR7','IL7R'],
    'memory_t':           ['GZMK','CD44','ID3','PRDM1','BCL6','BATF','CD27','CXCL13'],
    'tf_programs':        ['TBX21','GATA3','RORC','EOMES','BCL6','MAF','IRF4','BATF','PRDM1'],
    'tissue_resident':    ['ZNF683','ITGAE','CXCR6','PRDM1','CD69'],
    'stemness':           ['TCF7','LEF1','MYB','SELL','SLAMF6','BCL6','ID3'],
    'glycolysis':         ['HK1','HK2','PFKM','PKM','LDHA','ENO1','GAPDH','SLC2A1'],
    'oxphos':             ['NDUFA1','NDUFB1','SDHB','UQCRC1','COX4I1','ATP5F1A','ATP5F1B'],
    'mito_dysfunction':   ['MT-CO1','MT-ND1','MT-ND2','MT-CYB','MT-ATP6','PINK1','PRKN','TFAM'],
    'metabolic_recovery': ['TFAM','ESRRA','NRF1','SIRT3','PPARGC1A','CPT1A'],
    'apoptosis':          ['FAS','FASLG','BAK1','CASP3','CASP8','BCL2'],
    'senescence':         ['CDKN1A','CDKN2A','TP53','GLB1','IGFBP7','SERPINE1','RB1'],
    'cell_cycle':         ['CDK2','CDK4','CCND1','RB1','MKI67','TOP2A','PCNA','MCM2','CDK1','BIRC5'],
    'proliferation':      ['MKI67','TOP2A','PCNA','MCM2','CDK1'],
    'epigenetics':        ['DNMT1','DNMT3A','DNMT3B','TET2','EZH2','KDM6A','HDAC1','SIRT1'],
    'fibrosis':           ['TGFB1','TGFB2','TGFBR1','TGFBR2','COL1A1','COL3A1','ACTA2',
                           'FN1','TIMP1','MMP2'],
    'cancer_associated':  ['ATM','VIM','CDH1','SNAI1','ZEB1','MYC','CCND1','CDK4','CDKN2A'],
    'angiogenesis':       ['VEGFA','VEGFB','ANGPT2','PECAM1','FLT1','KDR','NRP1','HIF1A','EPAS1'],
}

# Check availability
gene_names = set(adata.var_names)
print(f"\n  Gene set availability:")
total_available = 0
total_defined = 0
for gs_name, gs_genes in gene_sets.items():
    avail = [g for g in gs_genes if g in gene_names]
    total_available += len(avail)
    total_defined += len(gs_genes)
    status = '✅' if len(avail) == len(gs_genes) else '⚠️'
    missing = [g for g in gs_genes if g not in gene_names]
    miss_str = f" missing: {missing}" if missing else ""
    print(f"  {status} {gs_name:20s} {len(avail):2d}/{len(gs_genes):2d}{miss_str}")

print(f"\n  Total: {total_available}/{total_defined} genes available")


  C4: 26 PATHWAY GENE SETS

  Gene set availability:
  ✅ inflammasome          6/ 6
  ✅ cytotoxicity          7/ 7
  ✅ checkpoint            6/ 6
  ✅ exhaustion            6/ 6
  ✅ nk_function           6/ 6
  ✅ nk_il15_dual          8/ 8
  ✅ il15_mtor            10/10
  ✅ immune_evasion        8/ 8
  ✅ treg                  5/ 5
  ✅ naive_t               5/ 5
  ✅ memory_t              8/ 8
  ✅ tf_programs           9/ 9
  ✅ tissue_resident       5/ 5
  ✅ stemness              7/ 7
  ✅ glycolysis            8/ 8
  ✅ oxphos                7/ 7
  ✅ mito_dysfunction      8/ 8
  ✅ metabolic_recovery    6/ 6
  ✅ apoptosis             6/ 6
  ✅ senescence            7/ 7
  ✅ cell_cycle           10/10
  ✅ proliferation         5/ 5
  ✅ epigenetics           8/ 8
  ✅ fibrosis             10/10
  ✅ cancer_associated     9/ 9
  ✅ angiogenesis          9/ 9

  Total: 189/189 genes available


In [18]:
# ============================================================
#  CELL 15: COMPUTE PATHWAY SCORES PER CELL (Mean Expression)
# ============================================================
print("=" * 70)
print("  COMPUTING PATHWAY SCORES (Mean Expression Proxy)")
print("  Method: mean(normalized expression) per gene set per cell")
print("  Validated: ρ = 0.922 concordance with AUCell (v17)")
print("=" * 70)

t0 = time.time()

# --- Split AnnData by Tissue (Fix for NameError) ---
print("  Splitting adata into Liver and Blood subsets...")
adata_liver = adata[adata.obs[tissue_col] == 'Liver'].copy()
adata_blood = adata[adata.obs[tissue_col] == 'Blood'].copy()
print(f"  adata_liver: {adata_liver.shape}")
print(f"  adata_blood: {adata_blood.shape}")

def compute_pathway_scores(ad, gene_sets_dict):
    """Compute mean expression-based pathway scores for each cell."""
    gene_names_ds = set(ad.var_names)
    for gs_name, gs_genes in gene_sets_dict.items():
        avail = [g for g in gs_genes if g in gene_names_ds]
        if len(avail) >= 2:
            gene_indices = [list(ad.var_names).index(g) for g in avail]
            # Extract expression data (handle sparse matrix)
            expr = ad.X[:, gene_indices]
            if hasattr(expr, 'toarray'):
                expr = expr.toarray()
            elif hasattr(expr, 'A'): # matrix subclass
                expr = expr.A

            # Compute mean per cell
            ad.obs[f'pw_{gs_name}'] = np.mean(expr, axis=1)
        else:
            ad.obs[f'pw_{gs_name}'] = 0.0
    return ad

adata_liver = compute_pathway_scores(adata_liver, gene_sets)
adata_blood = compute_pathway_scores(adata_blood, gene_sets)

print(f"  Computed {len(gene_sets)} pathway scores in {time.time()-t0:.1f}s")

# Verify
pw_cols = [c for c in adata_liver.obs.columns if c.startswith('pw_')]
print(f"  Pathway columns: {len(pw_cols)}")

  COMPUTING PATHWAY SCORES (Mean Expression Proxy)
  Method: mean(normalized expression) per gene set per cell
  Validated: ρ = 0.922 concordance with AUCell (v17)
  Splitting adata into Liver and Blood subsets...
  adata_liver: (106592, 24452)
  adata_blood: (136408, 24452)
  Computed 26 pathway scores in 17.1s
  Pathway columns: 26


In [20]:
# ============================================================
#  CELL 16: DONOR-LEVEL PATHWAY AGGREGATION & MWU TESTS
# ============================================================
print("=" * 70)
print("  DONOR-LEVEL PATHWAY SCORING (Tissue-Separated)")
print("=" * 70)

# Define Comparisons
COMPARISONS = [
    ('NL', 'IT', 'NL→IT'),
    ('NL', 'IA', 'NL→IA'),
    ('NL', 'AR', 'NL→AR'),
    ('NL', 'CR', 'NL→CR'),
    ('IT', 'IA', 'IT→IA'),
    ('IA', 'AR', 'IA→AR'),
    ('CR', 'AR', 'CR→AR')
]

def screen_pathways_tissue(ad, tissue_name, lineage_map, gene_sets, comparisons):
    """Screen all pathways × lineages × comparisons for one tissue."""
    results = []
    pw_names = list(gene_sets.keys())

    for lin_std, lin_data in lineage_map.items():
        lin_mask = ad.obs[lineage_col] == lin_data
        sub = ad[lin_mask]

        for pw_name in pw_names:
            pw_col = f'pw_{pw_name}'
            if pw_col not in sub.obs.columns:
                continue

            pw_vals = sub.obs[pw_col].values

            for s1, s2, comp_name in comparisons:
                # Donor-level aggregation
                def get_donor_means(stage):
                    stage_mask = sub.obs[stage_col].values == stage
                    donors = sub.obs.loc[stage_mask, donor_col].unique()
                    means = []
                    for d in donors:
                        d_mask = (sub.obs[stage_col].values == stage) & (sub.obs[donor_col].values == d)
                        if d_mask.sum() >= 5:
                            means.append(float(np.mean(pw_vals[d_mask])))
                    return means

                vals1 = get_donor_means(s1)
                vals2 = get_donor_means(s2)

                if len(vals1) < 2 or len(vals2) < 2:
                    continue

                mean1, mean2 = np.mean(vals1), np.mean(vals2)
                pct_change = ((mean2 - mean1) / abs(mean1) * 100) if abs(mean1) > 1e-10 else 0

                try:
                    stat, pval = mannwhitneyu(vals1, vals2, alternative='two-sided')
                except:
                    pval = 1.0

                n_pairs = len(vals1) * len(vals2)
                if mean2 >= mean1:
                    consist = sum(1 for v2 in vals2 for v1 in vals1 if v2 >= v1)
                else:
                    consist = sum(1 for v2 in vals2 for v1 in vals1 if v2 < v1)

                direction = '↑' if mean2 > mean1 else ('↓' if mean2 < mean1 else '=')
                sig = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else ('†' if pval<0.10 else 'NS')))

                results.append({
                    'tissue': tissue_name,
                    'lineage': lin_std,
                    'pathway': pw_name,
                    'comparison': comp_name,
                    'stage1': s1, 'stage2': s2,
                    'mean_s1': round(mean1, 6),
                    'mean_s2': round(mean2, 6),
                    'pct_change': round(pct_change, 1),
                    'direction': direction,
                    'p_value': round(pval, 6),
                    'sig': sig,
                    'consistency': f"{consist}/{n_pairs}",
                    'n_s1': len(vals1),
                    'n_s2': len(vals2),
                })

    return pd.DataFrame(results)

t0 = time.time()
df_pw_liver = screen_pathways_tissue(adata_liver, 'Liver', lineage_map, gene_sets, COMPARISONS)
print(f"  Liver: {len(df_pw_liver)} tests, {(df_pw_liver['p_value']<0.05).sum()} sig ({time.time()-t0:.0f}s)")

t0 = time.time()
df_pw_blood = screen_pathways_tissue(adata_blood, 'Blood', lineage_map, gene_sets, COMPARISONS)
print(f"  Blood: {len(df_pw_blood)} tests, {(df_pw_blood['p_value']<0.05).sum()} sig ({time.time()-t0:.0f}s)")

df_pw_liver.to_csv(f'{C4_SAVE}C4_pathway_liver.csv', index=False)
df_pw_blood.to_csv(f'{C4_SAVE}C4_pathway_blood.csv', index=False)
print(f"\n💾 Saved: C4_pathway_liver.csv, C4_pathway_blood.csv")

  DONOR-LEVEL PATHWAY SCORING (Tissue-Separated)
  Liver: 1274 tests, 108 sig (3s)
  Blood: 1274 tests, 130 sig (3s)

💾 Saved: C4_pathway_liver.csv, C4_pathway_blood.csv


In [22]:
# ============================================================
#  CELL 17: NL→IT PATHWAY CHANGES — Liver vs Blood SIMULTANEOUS
# ============================================================
print("=" * 70)
print("  🔴🔵 NL → IT: PATHWAY CHANGES (Liver vs Blood)")
print("=" * 70)

for lin in LINEAGES:  # Corrected from LINEAGES_STD to LINEAGES
    print(f"\n  ── {lin} ──")

    l_data = df_pw_liver[(df_pw_liver['comparison'] == 'NL→IT') & (df_pw_liver['lineage'] == lin)]
    b_data = df_pw_blood[(df_pw_blood['comparison'] == 'NL→IT') & (df_pw_blood['lineage'] == lin)]

    if len(l_data) == 0:
        print(f"    (no data)")
        continue

    merged = l_data[['pathway', 'pct_change', 'p_value', 'sig', 'direction']].merge(
        b_data[['pathway', 'pct_change', 'p_value', 'sig', 'direction']],
        on='pathway', suffixes=('_L', '_B')
    )

    # Show significant in either tissue first
    merged['any_sig'] = (merged['p_value_L'] < 0.05) | (merged['p_value_B'] < 0.05)
    merged = merged.sort_values(['any_sig', 'p_value_L'], ascending=[False, True])

    for _, row in merged.iterrows():
        if not row['any_sig']:
            continue

        l_mark = '★' if row['p_value_L'] < 0.05 else ('†' if row['p_value_L'] < 0.10 else ' ')
        b_mark = '★' if row['p_value_B'] < 0.05 else ('†' if row['p_value_B'] < 0.10 else ' ')
        match = '✅' if row['direction_L'] == row['direction_B'] else '⚠️↔'

        print(f"    {row['pathway']:22s} | 🔴{l_mark}{row['direction_L']}{abs(row['pct_change_L']):6.1f}% p={row['p_value_L']:.4f} | 🔵{b_mark}{row['direction_B']}{abs(row['pct_change_B']):6.1f}% p={row['p_value_B']:.4f} | {match}")

  🔴🔵 NL → IT: PATHWAY CHANGES (Liver vs Blood)

  ── Myeloid ──
    apoptosis              | 🔴★↑  95.4% p=0.0022 | 🔵★↑  66.8% p=0.0242 | ✅
    naive_t                | 🔴★↑  72.7% p=0.0043 | 🔵 ↓  10.3% p=0.6485 | ⚠️↔
    cancer_associated      | 🔴★↑   7.8% p=0.0260 | 🔵 ↑   1.5% p=0.5273 | ✅
    tissue_resident        | 🔴★↑  88.0% p=0.0411 | 🔵†↓  18.5% p=0.0727 | ⚠️↔
    il15_mtor              | 🔴†↑  21.0% p=0.0649 | 🔵★↑  57.7% p=0.0061 | ✅
    immune_evasion         | 🔴†↑  27.5% p=0.0649 | 🔵★↑  65.4% p=0.0061 | ✅
    epigenetics            | 🔴†↑  30.7% p=0.0931 | 🔵★↑  63.2% p=0.0061 | ✅
    metabolic_recovery     | 🔴 ↑  10.9% p=0.4848 | 🔵★↑  92.9% p=0.0121 | ✅
    inflammasome           | 🔴 ↑   6.6% p=0.5887 | 🔵★↑  51.3% p=0.0061 | ✅
    mito_dysfunction       | 🔴 ↑   2.3% p=0.9372 | 🔵★↑  17.2% p=0.0061 | ✅
    cytotoxicity           | 🔴 ↓   4.4% p=1.0000 | 🔵★↓  65.5% p=0.0424 | ✅

  ── CD4_T ──
    treg                   | 🔴★↑  74.2% p=0.0152 | 🔵 ↓  30.0% p=0.2677 | ⚠️↔
    exhaustion 

In [24]:
# ============================================================
#  CELL 18: ALL COMPARISONS — Significant Pathways Summary
# ============================================================
print("=" * 70)
print("  ALL COMPARISONS: Significant Pathways Count")
print("=" * 70)

print(f"\n  {'Lineage':<10s} {'Comparison':<10s} {'🔴Liver':>8s} {'🔵Blood':>8s} {'Both':>6s}")
print(f"  {'─'*48}")

for comp_name in [c[2] for c in COMPARISONS]:
    for lin in LINEAGES:  # Corrected from LINEAGES_STD to LINEAGES
        l_n = len(df_pw_liver[(df_pw_liver['comparison']==comp_name) & (df_pw_liver['lineage']==lin) & (df_pw_liver['p_value']<0.05)])
        b_n = len(df_pw_blood[(df_pw_blood['comparison']==comp_name) & (df_pw_blood['lineage']==lin) & (df_pw_blood['p_value']<0.05)])

        if l_n > 0 or b_n > 0:
            both = '✅' if l_n > 0 and b_n > 0 else ''
            print(f"  {lin:<10s} {comp_name:<10s} {l_n:>8d} {b_n:>8d} {both:>6s}")

  ALL COMPARISONS: Significant Pathways Count

  Lineage    Comparison   🔴Liver   🔵Blood   Both
  ────────────────────────────────────────────────
  Myeloid    NL→IT             4        8      ✅
  CD4_T      NL→IT             3        4      ✅
  CD8_T      NL→IT             6        4      ✅
  NK         NL→IT             1        6      ✅
  B          NL→IT             4        9      ✅
  PlasmaB    NL→IT             9        8      ✅
  Myeloid    NL→IA             2        4      ✅
  CD4_T      NL→IA             2        3      ✅
  CD8_T      NL→IA             2        2      ✅
  NK         NL→IA             2        8      ✅
  B          NL→IA             1        7      ✅
  PlasmaB    NL→IA             8        2      ✅
  Myeloid    NL→AR             9        7      ✅
  CD4_T      NL→AR             2        2      ✅
  CD8_T      NL→AR             4        1      ✅
  NK         NL→AR             2        6      ✅
  B          NL→AR             2       12      ✅
  PlasmaB    NL→AR  

In [27]:
# ============================================================
#  CELL 19: SELECT SIGNIFICANT PATHWAYS → Pass to C5
# ============================================================
print("=" * 70)
print("  🎯 SIGNIFICANT PATHWAY SELECTION FOR C5")
print("=" * 70)

# For each lineage, collect pathways significant in NL→IT (either tissue)
c5_selections = {}
for lin in LINEAGES:
    l_sig = set(df_pw_liver[(df_pw_liver['comparison']=='NL→IT') & (df_pw_liver['lineage']==lin) & (df_pw_liver['p_value']<0.05)]['pathway'])
    b_sig = set(df_pw_blood[(df_pw_blood['comparison']=='NL→IT') & (df_pw_blood['lineage']==lin) & (df_pw_blood['p_value']<0.05)]['pathway'])

    # Also include IT→IA and NL→CR significant
    l_sig2 = set(df_pw_liver[(df_pw_liver['comparison'].isin(['IT→IA','NL→CR'])) & (df_pw_liver['lineage']==lin) & (df_pw_liver['p_value']<0.05)]['pathway'])
    b_sig2 = set(df_pw_blood[(df_pw_blood['comparison'].isin(['IT→IA','NL→CR'])) & (df_pw_blood['lineage']==lin) & (df_pw_blood['p_value']<0.05)]['pathway'])

    all_sig = sorted(l_sig | b_sig | l_sig2 | b_sig2)
    c5_selections[lin] = all_sig

    print(f"\n  {lin}: {len(all_sig)} pathways")
    for pw in all_sig:
        in_l = '🔴' if pw in l_sig else '  '
        in_b = '🔵' if pw in b_sig else '  '
        print(f"    {in_l}{in_b} {pw}")

# Collect all genes from selected pathways
c5_genes = set()
for lin, pws in c5_selections.items():
    for pw in pws:
        if pw in gene_sets:
            c5_genes.update(gene_sets[pw])

c5_genes = sorted([g for g in c5_genes if g in gene_names])
print(f"\n  ▶ Total unique genes for C5: {len(c5_genes)}")

# Save
selection_rows = []
for lin, pws in c5_selections.items():
    for pw in pws:
        selection_rows.append({'lineage': lin, 'pathway': pw})
pd.DataFrame(selection_rows).to_csv(f'{C4_SAVE}C4_selected_pathways_for_C5.csv', index=False)
pd.DataFrame({'gene': c5_genes}).to_csv(f'{C4_SAVE}C4_selected_genes_for_C5.csv', index=False)

print(f"\n💾 Saved: C4_selected_pathways_for_C5.csv, C4_selected_genes_for_C5.csv")


  🎯 SIGNIFICANT PATHWAY SELECTION FOR C5

  Myeloid: 12 pathways
    🔴🔵 apoptosis
    🔴   cancer_associated
      🔵 cytotoxicity
      🔵 epigenetics
      🔵 il15_mtor
      🔵 immune_evasion
      🔵 inflammasome
      🔵 metabolic_recovery
      🔵 mito_dysfunction
    🔴   naive_t
         oxphos
    🔴   tissue_resident

  CD4_T: 8 pathways
      🔵 epigenetics
    🔴   exhaustion
         fibrosis
         metabolic_recovery
      🔵 mito_dysfunction
      🔵 tf_programs
    🔴🔵 tissue_resident
    🔴   treg

  CD8_T: 12 pathways
         angiogenesis
      🔵 cancer_associated
    🔴   checkpoint
    🔴   exhaustion
         fibrosis
    🔴   memory_t
    🔴🔵 metabolic_recovery
      🔵 mito_dysfunction
         nk_function
    🔴   stemness
      🔵 tissue_resident
    🔴   treg

  NK: 14 pathways
         apoptosis
         checkpoint
      🔵 epigenetics
      🔵 exhaustion
      🔵 fibrosis
      🔵 il15_mtor
         immune_evasion
         memory_t
      🔵 metabolic_recovery
      🔵 mito_dysfunction

In [28]:
# ============================================================
#  CELL 20: C4 SUMMARY
# ============================================================
print("=" * 70)
print("  ✅ C4 COMPLETE — Pathway Scoring Foundation")
print("=" * 70)

print(f"""
  Files saved to: {C4_SAVE}
  ├── C4_pathway_liver.csv          (all results, Liver)
  ├── C4_pathway_blood.csv          (all results, Blood)
  ├── C4_selected_pathways_for_C5.csv
  └── C4_selected_genes_for_C5.csv  ({len(c5_genes)} genes)

  ▶ NEXT: C5 — Individual Gene Markers for selected pathways
""")


  ✅ C4 COMPLETE — Pathway Scoring Foundation

  Files saved to: /content/drive/MyDrive/ITLAS/results/version18-analysis/C4_pathway/
  ├── C4_pathway_liver.csv          (all results, Liver)
  ├── C4_pathway_blood.csv          (all results, Blood)
  ├── C4_selected_pathways_for_C5.csv
  └── C4_selected_genes_for_C5.csv  (148 genes)

  ▶ NEXT: C5 — Individual Gene Markers for selected pathways



In [31]:
########################################################################
# V18-C5: INDIVIDUAL GENE MARKERS — Tissue-Separated, Liver & Blood
# ============================================================
# Step (3) of 7-step pipeline
# Prereq: C3 (proportions), C4 (pathways) completed
# Input:  c5_genes list from C4, adata_liver, adata_blood
# Output: version18-analysis/C5_genes/
########################################################################

# %% [markdown]
# # V18-C5: Individual Gene Markers (Tissue-Separated)
# **Step 3/7**: Analyze individual genes from significant pathways, Liver AND Blood simultaneously.



In [32]:
# ============================================================
#  CELL 21: SETUP C5
# ============================================================
C5_SAVE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/'
os.makedirs(C5_SAVE, exist_ok=True)

print("=" * 70)
print("  C5: INDIVIDUAL GENE MARKERS — Tissue-Separated")
print("=" * 70)

# Load selected genes from C4 (or use c5_genes from memory)
try:
    c5_gene_df = pd.read_csv(f'{C4_SAVE}C4_selected_genes_for_C5.csv')
    c5_genes = sorted(c5_gene_df['gene'].tolist())
except:
    print("  ⚠️ Using c5_genes from memory")

print(f"  Target genes: {len(c5_genes)}")

# Gene → pathway mapping
gene_to_pathway = {}
for gs_name, gs_genes in gene_sets.items():
    for g in gs_genes:
        if g in gene_to_pathway:
            gene_to_pathway[g] += f', {gs_name}'
        else:
            gene_to_pathway[g] = gs_name


  C5: INDIVIDUAL GENE MARKERS — Tissue-Separated
  Target genes: 148


In [33]:
# ============================================================
#  CELL 22: GENE SCREENING ENGINE (Tissue-Separated)
# ============================================================
print("=" * 70)
print("  GENE SCREENING ENGINE")
print("=" * 70)

def screen_gene_tissue(ad, tissue_name, gene, lin_data_name):
    """Screen one gene × one lineage across all comparisons in one tissue."""
    results = []

    lin_mask = ad.obs[lineage_col] == lin_data_name
    sub = ad[lin_mask]

    if gene not in sub.var_names:
        return results

    # Pre-extract expression
    gene_expr = sub[:, gene].X
    if hasattr(gene_expr, 'toarray'):
        gene_expr = gene_expr.toarray().flatten()
    else:
        gene_expr = np.array(gene_expr).flatten()

    for s1, s2, comp_name in COMPARISONS:
        def get_donor_means(stage):
            stage_mask = sub.obs[stage_col].values == stage
            donors = sub.obs.loc[stage_mask, donor_col].unique()
            means = []
            for d in donors:
                d_mask = (sub.obs[stage_col].values == stage) & (sub.obs[donor_col].values == d)
                if d_mask.sum() >= 5:
                    means.append(float(np.mean(gene_expr[d_mask])))
            return means

        vals1 = get_donor_means(s1)
        vals2 = get_donor_means(s2)

        if len(vals1) < 2 or len(vals2) < 2:
            continue

        mean1, mean2 = np.mean(vals1), np.mean(vals2)
        pct_change = ((mean2 - mean1) / abs(mean1) * 100) if abs(mean1) > 1e-10 else (99999 if mean2 > 0 else 0)

        try:
            stat, pval = mannwhitneyu(vals1, vals2, alternative='two-sided')
        except:
            pval = 1.0

        n_pairs = len(vals1) * len(vals2)
        if mean2 >= mean1:
            consist = sum(1 for v2 in vals2 for v1 in vals1 if v2 >= v1)
        else:
            consist = sum(1 for v2 in vals2 for v1 in vals1 if v2 < v1)

        direction = '↑' if mean2 > mean1 else ('↓' if mean2 < mean1 else '=')
        sig = '***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else ('†' if pval<0.10 else 'NS')))

        results.append({
            'tissue': tissue_name,
            'gene': gene,
            'lineage': lin_std,
            'pathway': gene_to_pathway.get(gene, 'other'),
            'comparison': comp_name,
            'stage1': s1, 'stage2': s2,
            'mean_s1': round(mean1, 6),
            'mean_s2': round(mean2, 6),
            'pct_change': round(pct_change, 1),
            'direction': direction,
            'p_value': round(pval, 6),
            'sig': sig,
            'consistency': f"{consist}/{n_pairs}",
            'consist_pct': round(consist/n_pairs*100, 1) if n_pairs > 0 else 0,
            'n_s1': len(vals1), 'n_s2': len(vals2),
        })

    return results

print(f"  Engine ready: {len(c5_genes)} genes × {len(lineage_map)} lineages × {len(COMPARISONS)} comparisons × 2 tissues")
print(f"  Expected tests: ~{len(c5_genes)*len(lineage_map)*len(COMPARISONS)*2:,}")


  GENE SCREENING ENGINE
  Engine ready: 148 genes × 7 lineages × 7 comparisons × 2 tissues
  Expected tests: ~14,504


In [34]:
# ============================================================
#  CELL 23: RUN LIVER SCREENING
# ============================================================
print("=" * 70)
print("  🔴 LIVER GENE SCREENING")
print("=" * 70)

results_liver = []
t0 = time.time()
total = len(c5_genes)

for gi, gene in enumerate(c5_genes):
    if (gi+1) % 20 == 0 or gi == 0:
        elapsed = time.time() - t0
        rate = (gi+1)/elapsed if elapsed > 0 else 1
        eta = (total-gi-1)/rate/60 if rate > 0 else 0
        print(f"  [{gi+1:3d}/{total}] {gene:12s} | {elapsed:.0f}s | ETA: {eta:.1f}min")

    for lin_std, lin_data in lineage_map.items():
        results_liver.extend(screen_gene_tissue(adata_liver, 'Liver', gene, lin_data))

df_gene_liver = pd.DataFrame(results_liver)
elapsed = time.time() - t0
print(f"\n✅ Liver: {len(df_gene_liver):,} tests, {(df_gene_liver['p_value']<0.05).sum()} sig ({elapsed:.0f}s)")
df_gene_liver.to_csv(f'{C5_SAVE}C5_genes_liver.csv', index=False)


  🔴 LIVER GENE SCREENING
  [  1/148] ACTA2        | 0s | ETA: 0.0min
  [ 20/148] CD44         | 10s | ETA: 1.0min
  [ 40/148] EOMES        | 20s | ETA: 0.9min
  [ 60/148] ID3          | 30s | ETA: 0.7min
  [ 80/148] MAF          | 41s | ETA: 0.6min
  [100/148] NLRP3        | 51s | ETA: 0.4min
  [120/148] SIRT1        | 61s | ETA: 0.2min
  [140/148] TP53         | 72s | ETA: 0.1min

✅ Liver: 7,252 tests, 398 sig (76s)


In [35]:
# ============================================================
#  CELL 24: RUN BLOOD SCREENING
# ============================================================
print("=" * 70)
print("  🔵 BLOOD GENE SCREENING")
print("=" * 70)

results_blood = []
t0 = time.time()

for gi, gene in enumerate(c5_genes):
    if (gi+1) % 20 == 0 or gi == 0:
        elapsed = time.time() - t0
        rate = (gi+1)/elapsed if elapsed > 0 else 1
        eta = (total-gi-1)/rate/60 if rate > 0 else 0
        print(f"  [{gi+1:3d}/{total}] {gene:12s} | {elapsed:.0f}s | ETA: {eta:.1f}min")

    for lin_std, lin_data in lineage_map.items():
        results_blood.extend(screen_gene_tissue(adata_blood, 'Blood', gene, lin_data))

df_gene_blood = pd.DataFrame(results_blood)
elapsed = time.time() - t0
print(f"\n✅ Blood: {len(df_gene_blood):,} tests, {(df_gene_blood['p_value']<0.05).sum()} sig ({elapsed:.0f}s)")
df_gene_blood.to_csv(f'{C5_SAVE}C5_genes_blood.csv', index=False)


  🔵 BLOOD GENE SCREENING
  [  1/148] ACTA2        | 0s | ETA: 0.0min
  [ 20/148] CD44         | 11s | ETA: 1.2min
  [ 40/148] EOMES        | 23s | ETA: 1.0min
  [ 60/148] ID3          | 35s | ETA: 0.9min
  [ 80/148] MAF          | 47s | ETA: 0.7min
  [100/148] NLRP3        | 59s | ETA: 0.5min
  [120/148] SIRT1        | 70s | ETA: 0.3min
  [140/148] TP53         | 82s | ETA: 0.1min

✅ Blood: 7,252 tests, 610 sig (88s)


In [39]:
# ============================================================
#  CELL 26: NL→IT SIGNIFICANT GENES — Liver vs Blood SIMULTANEOUS
# ============================================================
print("=" * 70)
print("  🔴🔵 NL → IT: SIGNIFICANT GENES (Simultaneous)")
print("=" * 70)

# Prepare discrepancy dataframe (disc)
l_dat = df_gene_liver[df_gene_liver['comparison'] == 'NL→IT'].copy()
b_dat = df_gene_blood[df_gene_blood['comparison'] == 'NL→IT'].copy()

disc = pd.merge(l_dat, b_dat, on=['lineage', 'gene'], suffixes=('_L', '_B'))
disc = disc.rename(columns={
    'p_value_L': 'L_p', 'p_value_B': 'B_p',
    'direction_L': 'L_dir', 'direction_B': 'B_dir',
    'pct_change_L': 'L_pct', 'pct_change_B': 'B_pct'
})
disc['dir_same'] = (disc['L_dir'] == disc['B_dir'])

for lin in LINEAGES:
    print(f"\n  ── {lin} ──")
    lin_disc = disc[(disc['lineage']==lin) & ((disc['L_p']<0.05)|(disc['B_p']<0.05))]
    lin_disc = lin_disc.sort_values('L_p')

    if len(lin_disc) == 0:
        print(f"    (no significant genes)")
        continue

    for _, row in lin_disc.iterrows():
        l_m = '★' if row['L_p']<0.05 else ('†' if row['L_p']<0.10 else ' ')
        b_m = '★' if row['B_p']<0.05 else ('†' if row['B_p']<0.10 else ' ')
        match = '✅' if row['dir_same'] else '⚠️↔'
        pw = gene_to_pathway.get(row['gene'], '')[:20]

        print(f"    {row['gene']:12s} | 🔴{l_m}{row['L_dir']}{abs(row['L_pct']):7.1f}% p={row['L_p']:.4f} | 🔵{b_m}{row['B_dir']}{abs(row['B_pct']):7.1f}% p={row['B_p']:.4f} | {match} [{pw}]")

  🔴🔵 NL → IT: SIGNIFICANT GENES (Simultaneous)

  ── Myeloid ──
    BAK1         | 🔴★↑  379.5% p=0.0022 | 🔵★↑  161.9% p=0.0061 | ✅ [apoptosis]
    DNMT1        | 🔴★↑  125.2% p=0.0022 | 🔵★↑  120.9% p=0.0061 | ✅ [epigenetics]
    TOX          | 🔴★↑ 1856.8% p=0.0043 | 🔵 ↓   92.9% p=0.7662 | ⚠️↔ [exhaustion]
    PRF1         | 🔴★↑99999.0% p=0.0096 | 🔵 ↓   19.4% p=0.6989 | ⚠️↔ [cytotoxicity, nk_il1]
    CCR7         | 🔴★↑  460.6% p=0.0129 | 🔵 ↑  629.5% p=0.1218 | ✅ [naive_t]
    IRF4         | 🔴★↑  231.0% p=0.0152 | 🔵★↑ 1644.1% p=0.0100 | ✅ [tf_programs]
    CASP3        | 🔴★↑  112.4% p=0.0260 | 🔵★↑  103.3% p=0.0242 | ✅ [apoptosis]
    BATF         | 🔴★↓   62.4% p=0.0260 | 🔵 ↑    6.8% p=0.7879 | ⚠️↔ [exhaustion, memory_t]
    ATM          | 🔴★↑   43.8% p=0.0260 | 🔵†↑   62.4% p=0.0727 | ✅ [cancer_associated]
    NCR1         | 🔴★↑99999.0% p=0.0284 | 🔵 ↑99999.0% p=0.3266 | ✅ [nk_function]
    MTOR         | 🔴★↑  327.3% p=0.0341 | 🔵★↑  432.7% p=0.0061 | ✅ [il15_mtor]
    BIRC5        | 🔴★↑  51

In [41]:
# ============================================================
#  CELL 27: SELECT SIGNIFICANT GENES → Pass to C6/C7
# ============================================================
print("=" * 70)
print("  🎯 SIGNIFICANT GENE SELECTION FOR C6/C7")
print("=" * 70)

# All comparisons, collect significant genes per lineage per tissue
sig_gene_summary = []

for comp_name in [c[2] for c in COMPARISONS]:
    for lin in LINEAGES:
        l_sig = df_gene_liver[(df_gene_liver['comparison']==comp_name) & (df_gene_liver['lineage']==lin) & (df_gene_liver['p_value']<0.05)]
        b_sig = df_gene_blood[(df_gene_blood['comparison']==comp_name) & (df_gene_blood['lineage']==lin) & (df_gene_blood['p_value']<0.05)]

        for _, row in l_sig.iterrows():
            sig_gene_summary.append({**row.to_dict(), 'sig_tissue': 'Liver'})
        for _, row in b_sig.iterrows():
            sig_gene_summary.append({**row.to_dict(), 'sig_tissue': 'Blood'})

df_sig_all = pd.DataFrame(sig_gene_summary)
df_sig_all.to_csv(f'{C5_SAVE}C5_all_significant_genes.csv', index=False)

# Top genes by frequency of significance across lineages/comparisons
gene_sig_counts = df_sig_all.groupby('gene').agg(
    n_sig=('comparison', 'count'),
    n_lineages=('lineage', 'nunique'),
    n_comparisons=('comparison', 'nunique'),
    tissues=('sig_tissue', lambda x: ','.join(sorted(set(x)))),
).sort_values('n_sig', ascending=False)

print(f"\n  Top 20 Most Frequently Significant Genes:")
print(f"  {'Gene':12s} {'#Sig':>5s} {'#Lin':>5s} {'#Comp':>6s} {'Tissues'}")
for gene, row in gene_sig_counts.head(20).iterrows():
    pw = gene_to_pathway.get(gene, '')[:25]
    print(f"  {gene:12s} {row['n_sig']:5d} {row['n_lineages']:5d} {row['n_comparisons']:6d} {row['tissues']:15s} [{pw}]")

gene_sig_counts.to_csv(f'{C5_SAVE}C5_gene_significance_ranking.csv')


  🎯 SIGNIFICANT GENE SELECTION FOR C6/C7

  Top 20 Most Frequently Significant Genes:
  Gene          #Sig  #Lin  #Comp Tissues
  JAK1            32     6      4 Blood,Liver     [il15_mtor]
  MT-CYB          26     6      4 Blood,Liver     [mito_dysfunction]
  TFAM            24     5      4 Blood,Liver     [mito_dysfunction, metabol]
  TGFBR2          23     6      4 Blood,Liver     [fibrosis]
  MT-ND2          23     6      5 Blood,Liver     [mito_dysfunction]
  RPTOR           21     6      5 Blood,Liver     [il15_mtor]
  SDHB            20     6      4 Blood,Liver     [oxphos]
  MT-ND1          20     6      4 Blood,Liver     [mito_dysfunction]
  DNMT3A          18     6      4 Blood,Liver     [epigenetics]
  GZMK            15     6      6 Blood,Liver     [cytotoxicity, memory_t]
  CD27            15     6      6 Blood,Liver     [memory_t]
  DNMT1           14     5      5 Blood,Liver     [epigenetics]
  IL2RA           14     5      6 Blood,Liver     [treg]
  CD44            14  

In [42]:
# ============================================================
#  CELL 28: C5 COMPLETE
# ============================================================
print("=" * 70)
print("  ✅ C5 COMPLETE — Individual Gene Markers")
print("=" * 70)

print(f"""
  Files saved to: {C5_SAVE}
  ├── C5_genes_liver.csv            (all gene results, Liver)
  ├── C5_genes_blood.csv            (all gene results, Blood)
  ├── C5_tissue_discrepancy_NL_IT.csv
  ├── C5_all_significant_genes.csv
  └── C5_gene_significance_ranking.csv

  ▶ NEXT: C6 — Tissue Discrepancy Master Compilation
  ▶ THEN: C7 — Inter-relationship Pattern Discovery
""")


  ✅ C5 COMPLETE — Individual Gene Markers

  Files saved to: /content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/
  ├── C5_genes_liver.csv            (all gene results, Liver)
  ├── C5_genes_blood.csv            (all gene results, Blood)
  ├── C5_tissue_discrepancy_NL_IT.csv
  ├── C5_all_significant_genes.csv
  └── C5_gene_significance_ranking.csv

  ▶ NEXT: C6 — Tissue Discrepancy Master Compilation
  ▶ THEN: C7 — Inter-relationship Pattern Discovery



In [51]:
########################################################################
# V18-C6/C7/C8: TISSUE DISCREPANCY → PATTERNS → DISEASE CHARACTERISTICS
# ============================================================
# Steps (4), (5), (6) of 7-step pipeline
# Prereq: C3-C5 completed
# Output: version18-analysis/C6_discrepancy/, C7_patterns/, C8_characteristics/
########################################################################

# %% [markdown]
# # V18-C6: Tissue Discrepancy Master Compilation (Step 4/7)
# # V18-C7: Inter-relationship Pattern Discovery (Step 5/7)
# # V18-C8: Disease Spectrum-Specific Characteristics (Step 6/7)


In [53]:
# ============================================================
#  C6: TISSUE DISCREPANCY MASTER COMPILATION
#  = User Step (4)
# ============================================================
print("=" * 70)
print("  C6: MASTER TISSUE DISCREPANCY COMPILATION")
print("=" * 70)

C6_SAVE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C6_discrepancy/'
os.makedirs(C6_SAVE, exist_ok=True)

# Load C3-C5 results
# df_mwu_liver/blood (from C3), df_pw_liver/blood (from C4), df_gene_liver/blood (from C5)

# ── C6-1: Proportion discrepancy (Skipped as df_simul is missing) ──
print("\n  ── C6-1: Proportion Discrepancy (Skipped) ──")
prop_disc = pd.DataFrame() # Empty placeholder

# ── C6-2: Pathway discrepancy ──
print("  ── C6-2: Pathway Discrepancy ──")
pw_disc_rows = []
for comp_name in [c[2] for c in COMPARISONS]:
    for lin in LINEAGES: # Corrected from LINEAGES_STD
        l_pws = df_pw_liver[(df_pw_liver['comparison']==comp_name) & (df_pw_liver['lineage']==lin)]
        b_pws = df_pw_blood[(df_pw_blood['comparison']==comp_name) & (df_pw_blood['lineage']==lin)]

        merged = l_pws[['pathway','pct_change','p_value','direction']].merge(
            b_pws[['pathway','pct_change','p_value','direction']],
            on='pathway', suffixes=('_L','_B'), how='outer'
        )

        for _, row in merged.iterrows():
            pw_disc_rows.append({
                'level': 'pathway',
                'item': f"{lin}/{row['pathway']}",
                'lineage': lin,
                'comparison': comp_name,
                'L_pct': row.get('pct_change_L', np.nan),
                'B_pct': row.get('pct_change_B', np.nan),
                'L_p': row.get('p_value_L', np.nan),
                'B_p': row.get('p_value_B', np.nan),
                'L_dir': row.get('direction_L', ''),
                'B_dir': row.get('direction_B', ''),
                'dir_same': row.get('direction_L','') == row.get('direction_B',''),
                'both_sig': (row.get('p_value_L',1) < 0.05) and (row.get('p_value_B',1) < 0.05),
                'liver_only': (row.get('p_value_L',1) < 0.05) and (row.get('p_value_B',1) >= 0.05),
                'blood_only': (row.get('p_value_L',1) >= 0.05) and (row.get('p_value_B',1) < 0.05),
            })

df_pw_disc = pd.DataFrame(pw_disc_rows)

# ── C6-3: Gene discrepancy ── (already computed in C5 for NL→IT; extend to all comparisons)
print("  ── C6-3: Gene Discrepancy (all comparisons) ──")
gene_disc_rows = []
for comp_name in [c[2] for c in COMPARISONS]:
    l_genes = df_gene_liver[df_gene_liver['comparison']==comp_name][['gene','lineage','pct_change','p_value','direction']].copy()
    l_genes.columns = ['gene','lineage','L_pct','L_p','L_dir']

    b_genes = df_gene_blood[df_gene_blood['comparison']==comp_name][['gene','lineage','pct_change','p_value','direction']].copy()
    b_genes.columns = ['gene','lineage','B_pct','B_p','B_dir']

    merged = l_genes.merge(b_genes, on=['gene','lineage'], how='outer')
    merged['comparison'] = comp_name
    merged['level'] = 'gene'
    merged['item'] = merged['gene'] + '/' + merged['lineage']
    merged['dir_same'] = merged['L_dir'] == merged['B_dir']
    merged['both_sig'] = (merged['L_p'] < 0.05) & (merged['B_p'] < 0.05)
    merged['liver_only'] = (merged['L_p'] < 0.05) & (merged['B_p'] >= 0.05)
    merged['blood_only'] = (merged['L_p'] >= 0.05) & (merged['B_p'] < 0.05)

    gene_disc_rows.append(merged)

df_gene_disc = pd.concat(gene_disc_rows, ignore_index=True)

# ── C6-4: Master summary ──
print("\n  ── MASTER DISCREPANCY SUMMARY ──")

for level, df in [('Proportion', prop_disc), # Removed condition
                   ('Pathway', df_pw_disc),
                   ('Gene', df_gene_disc)]:
    if len(df) == 0:
        continue

    total = len(df)
    dir_same = df['dir_same'].sum() if 'dir_same' in df else 0
    both = df['both_sig'].sum() if 'both_sig' in df else 0
    l_only = df['liver_only'].sum() if 'liver_only' in df else 0
    b_only = df['blood_only'].sum() if 'blood_only' in df else 0

    print(f"\n  {level} Level:")
    print(f"    Total pairs: {total}")
    print(f"    Direction concordant: {dir_same} ({dir_same/total*100:.0f}%)")
    print(f"    Both significant: {both}")
    print(f"    Liver-only sig: {l_only}")
    print(f"    Blood-only sig: {b_only}")

# Save master file
df_gene_disc.to_csv(f'{C6_SAVE}C6_gene_discrepancy_master.csv', index=False)
df_pw_disc.to_csv(f'{C6_SAVE}C6_pathway_discrepancy_master.csv', index=False)
print(f"\n💾 Saved: C6_gene_discrepancy_master.csv, C6_pathway_discrepancy_master.csv")

  C6: MASTER TISSUE DISCREPANCY COMPILATION

  ── C6-1: Proportion Discrepancy (Skipped) ──
  ── C6-2: Pathway Discrepancy ──
  ── C6-3: Gene Discrepancy (all comparisons) ──

  ── MASTER DISCREPANCY SUMMARY ──

  Pathway Level:
    Total pairs: 1092
    Direction concordant: 667 (61%)
    Both significant: 28
    Liver-only sig: 73
    Blood-only sig: 102

  Gene Level:
    Total pairs: 7252
    Direction concordant: 4140 (57%)
    Both significant: 83
    Liver-only sig: 315
    Blood-only sig: 527

💾 Saved: C6_gene_discrepancy_master.csv, C6_pathway_discrepancy_master.csv


In [54]:
# ============================================================
#  C7: INTER-RELATIONSHIP PATTERN DISCOVERY
#  = User Step (5)
# ============================================================
print("=" * 70)
print("  C7: INTER-RELATIONSHIP PATTERN DISCOVERY")
print("=" * 70)

C7_SAVE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C7_patterns/'
os.makedirs(C7_SAVE, exist_ok=True)

# ── C7-1: Select top genes (most frequently significant) ──
print("  ── C7-1: Top Gene Selection ──")

# Combine liver and blood significant genes
all_sig = pd.concat([
    df_gene_liver[df_gene_liver['p_value']<0.05].assign(sig_tissue='Liver'),
    df_gene_blood[df_gene_blood['p_value']<0.05].assign(sig_tissue='Blood'),
])

gene_freq = all_sig.groupby('gene').agg(
    n_sig=('comparison', 'count'),
    n_lineages=('lineage', 'nunique'),
    n_comparisons=('comparison', 'nunique'),
    lineages=('lineage', lambda x: ','.join(sorted(set(x)))),
    tissues=('sig_tissue', lambda x: ','.join(sorted(set(x)))),
    comps=('comparison', lambda x: ','.join(sorted(set(x)))),
).sort_values('n_sig', ascending=False)

# Select top 10-15 genes
TOP_N = min(15, len(gene_freq))
top_genes = gene_freq.head(TOP_N).index.tolist()

print(f"\n  Top {TOP_N} genes selected for pattern analysis:")
for i, gene in enumerate(top_genes):
    row = gene_freq.loc[gene]
    pw = gene_to_pathway.get(gene, '')[:30]
    print(f"  {i+1:2d}. {gene:12s} {row['n_sig']:3d}sig in {row['n_lineages']}lin/{row['n_comparisons']}comp | {row['tissues']:15s} [{pw}]")

pd.DataFrame({'rank': range(1, TOP_N+1), 'gene': top_genes}).to_csv(f'{C7_SAVE}C7_top_genes.csv', index=False)


  C7: INTER-RELATIONSHIP PATTERN DISCOVERY
  ── C7-1: Top Gene Selection ──

  Top 15 genes selected for pattern analysis:
   1. JAK1          32sig in 6lin/4comp | Blood,Liver     [il15_mtor]
   2. MT-CYB        27sig in 7lin/4comp | Blood,Liver     [mito_dysfunction]
   3. TFAM          25sig in 6lin/4comp | Blood,Liver     [mito_dysfunction, metabolic_re]
   4. TGFBR2        23sig in 6lin/4comp | Blood,Liver     [fibrosis]
   5. MT-ND2        23sig in 6lin/5comp | Blood,Liver     [mito_dysfunction]
   6. RPTOR         21sig in 6lin/5comp | Blood,Liver     [il15_mtor]
   7. SDHB          20sig in 6lin/4comp | Blood,Liver     [oxphos]
   8. MT-ND1        20sig in 6lin/4comp | Blood,Liver     [mito_dysfunction]
   9. DNMT3A        18sig in 6lin/4comp | Blood,Liver     [epigenetics]
  10. CD27          17sig in 7lin/6comp | Blood,Liver     [memory_t]
  11. GZMK          16sig in 7lin/6comp | Blood,Liver     [cytotoxicity, memory_t]
  12. DNMT1         14sig in 5lin/5comp | Blood,Liver  

In [55]:
# ── C7-2: Gene-to-Gene Correlation (within lineage) ──
print("\n  ── C7-2: Gene-to-Gene Correlations ──")
from scipy.stats import spearmanr

def compute_gene_gene_corr(ad, tissue_name, genes, lin_data):
    """Donor-level Spearman correlation between gene pairs."""
    lin_mask = ad.obs[lineage_col] == lin_data
    sub = ad[lin_mask]

    # Donor-level means for each gene
    donors = sub.obs[donor_col].unique()
    donor_means = {}

    for gene in genes:
        if gene not in sub.var_names:
            continue
        expr = sub[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = np.array(expr).flatten()

        d_means = []
        for d in donors:
            d_mask = sub.obs[donor_col].values == d
            if d_mask.sum() >= 5:
                d_means.append(float(np.mean(expr[d_mask])))
            else:
                d_means.append(np.nan)
        donor_means[gene] = d_means

    # Pairwise correlations
    results = []
    gene_list = [g for g in genes if g in donor_means]

    for i in range(len(gene_list)):
        for j in range(i+1, len(gene_list)):
            g1, g2 = gene_list[i], gene_list[j]
            v1 = np.array(donor_means[g1])
            v2 = np.array(donor_means[g2])

            # Remove NaN
            valid = ~np.isnan(v1) & ~np.isnan(v2)
            if valid.sum() < 5:
                continue

            rho, pval = spearmanr(v1[valid], v2[valid])
            results.append({
                'tissue': tissue_name,
                'lineage': lin_std,
                'gene1': g1, 'gene2': g2,
                'spearman_rho': round(rho, 4),
                'p_value': round(pval, 6),
                'n_donors': int(valid.sum()),
                'sig': '*' if pval < 0.05 else 'NS',
            })

    return results

corr_results = []
for lin_std, lin_data in lineage_map.items():
    for tissue_name, ad in [('Liver', adata_liver), ('Blood', adata_blood)]:
        corr_results.extend(compute_gene_gene_corr(ad, tissue_name, top_genes, lin_data))

df_corr = pd.DataFrame(corr_results)
if len(df_corr) > 0:
    sig_corr = df_corr[df_corr['p_value'] < 0.05]
    print(f"\n  Total correlations: {len(df_corr)}, Significant: {len(sig_corr)}")

    # Show strongest correlations
    print(f"\n  Top 20 strongest significant correlations:")
    for _, row in sig_corr.sort_values('spearman_rho', ascending=False).head(20).iterrows():
        print(f"    {row['gene1']:10s}↔{row['gene2']:10s} ρ={row['spearman_rho']:+.3f} p={row['p_value']:.4f} | {row['tissue']}/{row['lineage']}")

    df_corr.to_csv(f'{C7_SAVE}C7_gene_gene_correlations.csv', index=False)



  ── C7-2: Gene-to-Gene Correlations ──

  Total correlations: 1470, Significant: 517

  Top 20 strongest significant correlations:
    MT-CYB    ↔MT-ND1     ρ=+0.929 p=0.0000 | Blood/CD8_T
    MT-CYB    ↔MT-ND1     ρ=+0.887 p=0.0000 | Blood/CD4_T
    JAK1      ↔TGFBR2     ρ=+0.883 p=0.0000 | Blood/CD4_T
    MT-CYB    ↔MT-ND1     ρ=+0.879 p=0.0000 | Blood/NK
    JAK1      ↔TGFBR2     ρ=+0.871 p=0.0000 | Liver/CD4_T
    TFAM      ↔DNMT1      ρ=+0.857 p=0.0000 | Blood/Myeloid
    JAK1      ↔TGFBR2     ρ=+0.849 p=0.0000 | Blood/NK
    MT-ND2    ↔MT-ND1     ρ=+0.848 p=0.0000 | Blood/NK
    IL2RA     ↔MTOR       ρ=+0.838 p=0.0002 | Blood/gdT
    MT-CYB    ↔MT-ND2     ρ=+0.833 p=0.0000 | Blood/CD8_T
    TFAM      ↔DNMT1      ρ=+0.832 p=0.0000 | Blood/NK
    TFAM      ↔MTOR       ρ=+0.832 p=0.0000 | Blood/B
    JAK1      ↔TGFBR2     ρ=+0.830 p=0.0000 | Blood/B
    MT-ND2    ↔MT-ND1     ρ=+0.823 p=0.0000 | Blood/CD8_T
    MT-CYB    ↔MT-ND2     ρ=+0.821 p=0.0000 | Blood/CD4_T
    MT-CYB    ↔MT

In [57]:
# ── C7-3: Cell-to-Cell Patterns (same gene, multiple lineages) ──
print("\n  ── C7-3: Cell-to-Cell Patterns (Cross-Lineage) ──")

for gene in top_genes[:10]:
    l_data = df_gene_liver[(df_gene_liver['gene']==gene) & (df_gene_liver['comparison']=='NL→IT')]
    b_data = df_gene_blood[(df_gene_blood['gene']==gene) & (df_gene_blood['comparison']=='NL→IT')]

    l_sig = l_data[l_data['p_value'] < 0.05]
    b_sig = b_data[b_data['p_value'] < 0.05]

    if len(l_sig) >= 2 or len(b_sig) >= 2:  # Multi-lineage
        print(f"\n  {gene} (NL→IT):")
        for lin in LINEAGES:
            lr = l_data[l_data['lineage']==lin]
            br = b_data[b_data['lineage']==lin]
            if len(lr) > 0 and len(br) > 0:
                lr, br = lr.iloc[0], br.iloc[0]
                l_m = '★' if lr['p_value']<0.05 else ' '
                b_m = '★' if br['p_value']<0.05 else ' '
                print(f"    {lin:10s} 🔴{l_m}{lr['direction']}{abs(lr['pct_change']):6.1f}% 🔵{b_m}{br['direction']}{abs(br['pct_change']):6.1f}%")




  ── C7-3: Cell-to-Cell Patterns (Cross-Lineage) ──

  JAK1 (NL→IT):
    Myeloid    🔴 ↑  31.2% 🔵★↑  88.0%
    CD4_T      🔴★↑  36.0% 🔵★↑  23.1%
    CD8_T      🔴 ↑  18.4% 🔵 ↑  27.7%
    NK         🔴 ↑  21.0% 🔵★↑  50.5%
    B          🔴★↑  73.1% 🔵★↑  49.5%
    PlasmaB    🔴★↑  76.1% 🔵★↑ 128.8%

  MT-CYB (NL→IT):
    Myeloid    🔴 ↑   5.9% 🔵★↑  14.9%
    CD4_T      🔴★↑  13.1% 🔵★↑  23.1%
    CD8_T      🔴 ↑  12.7% 🔵★↑  24.7%
    NK         🔴 ↑   9.7% 🔵★↑  24.0%
    B          🔴 ↑   2.5% 🔵★↑   8.3%
    PlasmaB    🔴★↑  26.3% 🔵 ↑  15.5%

  TFAM (NL→IT):
    Myeloid    🔴 ↑  18.4% 🔵★↑ 120.2%
    CD4_T      🔴★↑  42.7% 🔵★↑  97.4%
    CD8_T      🔴★↑  55.3% 🔵★↑  93.0%
    NK         🔴★↑  53.0% 🔵★↑  79.2%
    B          🔴 ↓  18.1% 🔵★↑ 103.2%
    PlasmaB    🔴 ↑  75.9% 🔵 ↓   2.5%

  TGFBR2 (NL→IT):
    Myeloid    🔴 ↑   3.3% 🔵★↑  48.7%
    CD4_T      🔴★↑  46.3% 🔵★↑  73.8%
    CD8_T      🔴 ↑  35.9% 🔵★↑  97.5%
    NK         🔴 ↑  31.6% 🔵★↑  80.9%
    B          🔴 ↑  82.8% 🔵★↑  62.8%
    PlasmaB    🔴 ↑ 128.2

In [58]:
# ============================================================
#  C8: DISEASE SPECTRUM-SPECIFIC CHARACTERISTICS
#  = User Step (6)
# ============================================================
print("=" * 70)
print("  C8: DISEASE SPECTRUM-SPECIFIC CHARACTERISTICS")
print("=" * 70)

C8_SAVE = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics/'
os.makedirs(C8_SAVE, exist_ok=True)

# ── IT-Specific: NL→IT sig, NL→IA NS ──
print("\n  ── IT-SPECIFIC GENES (NL→IT sig, NL→IA NS) ──")
for tissue_name, df in [('Liver', df_gene_liver), ('Blood', df_gene_blood)]:
    nl_it = df[df['comparison']=='NL→IT'].set_index(['gene','lineage'])
    nl_ia = df[df['comparison']=='NL→IA'].set_index(['gene','lineage'])

    common = nl_it.index.intersection(nl_ia.index)
    it_specific = []
    for idx in common:
        if nl_it.loc[idx, 'p_value'] < 0.05 and nl_ia.loc[idx, 'p_value'] > 0.10:
            it_specific.append({
                'gene': idx[0], 'lineage': idx[1], 'tissue': tissue_name,
                'IT_pct': nl_it.loc[idx, 'pct_change'],
                'IT_p': nl_it.loc[idx, 'p_value'],
                'IA_p': nl_ia.loc[idx, 'p_value'],
            })

    df_it_spec = pd.DataFrame(it_specific)
    if len(df_it_spec) > 0:
        print(f"\n  {tissue_name}: {len(df_it_spec)} IT-specific genes")
        for _, row in df_it_spec.sort_values('IT_p').head(10).iterrows():
            print(f"    {row['gene']:12s} {row['lineage']:8s} IT:{row['IT_pct']:+.1f}% p={row['IT_p']:.4f} | IA:p={row['IA_p']:.3f}(NS)")

# ── IA Transition: IT→IA sig ──
print("\n  ── IA TRANSITION GENES (IT→IA sig) ──")
for tissue_name, df in [('Liver', df_gene_liver), ('Blood', df_gene_blood)]:
    it_ia_sig = df[(df['comparison']=='IT→IA') & (df['p_value']<0.05)]
    if len(it_ia_sig) > 0:
        print(f"\n  {tissue_name}: {len(it_ia_sig)} IT→IA transition genes")
        for _, row in it_ia_sig.sort_values('p_value').head(10).iterrows():
            print(f"    {row['gene']:12s} {row['lineage']:8s} {row['direction']}{abs(row['pct_change']):.1f}% p={row['p_value']:.4f}")

# ── CR Scar: NL→CR sig, NL→IA NS ──
print("\n  ── CR SCAR GENES (NL→CR sig, NL→IA NS) ──")
for tissue_name, df in [('Liver', df_gene_liver), ('Blood', df_gene_blood)]:
    nl_cr = df[df['comparison']=='NL→CR'].set_index(['gene','lineage'])
    nl_ia = df[df['comparison']=='NL→IA'].set_index(['gene','lineage'])

    common = nl_cr.index.intersection(nl_ia.index)
    cr_scar = []
    for idx in common:
        if nl_cr.loc[idx, 'p_value'] < 0.05 and nl_ia.loc[idx, 'p_value'] > 0.10:
            cr_scar.append({
                'gene': idx[0], 'lineage': idx[1], 'tissue': tissue_name,
                'CR_pct': nl_cr.loc[idx, 'pct_change'],
                'CR_p': nl_cr.loc[idx, 'p_value'],
                'IA_p': nl_ia.loc[idx, 'p_value'],
            })

    df_cr_scar = pd.DataFrame(cr_scar)
    if len(df_cr_scar) > 0:
        print(f"\n  {tissue_name}: {len(df_cr_scar)} CR scar genes")
        for _, row in df_cr_scar.sort_values('CR_p').head(10).iterrows():
            print(f"    {row['gene']:12s} {row['lineage']:8s} CR:{row['CR_pct']:+.1f}% p={row['CR_p']:.4f} | IA:p={row['IA_p']:.3f}(NS)")

# ── IA vs AR: What chronic fails but acute achieves ──
print("\n  ── IA vs AR DISCRIMINATORS (chronic vs acute resolution) ──")
for tissue_name, df in [('Liver', df_gene_liver), ('Blood', df_gene_blood)]:
    ia_ar_sig = df[(df['comparison']=='IA→AR') & (df['p_value']<0.05)]
    if len(ia_ar_sig) > 0:
        print(f"\n  {tissue_name}: {len(ia_ar_sig)} IA-AR discriminators")
        for _, row in ia_ar_sig.sort_values('p_value').head(10).iterrows():
            # AR direction: ↑AR means AR higher, which could be recovery
            ar_label = '↑AR' if row['direction'] == '↑' else '↓AR'
            print(f"    {row['gene']:12s} {row['lineage']:8s} {ar_label}{abs(row['pct_change']):.1f}% p={row['p_value']:.4f}")


  C8: DISEASE SPECTRUM-SPECIFIC CHARACTERISTICS

  ── IT-SPECIFIC GENES (NL→IT sig, NL→IA NS) ──

  Liver: 37 IT-specific genes
    DNMT1        Myeloid  IT:+125.2% p=0.0022 | IA:p=0.329(NS)
    TFAM         CD8_T    IT:+55.3% p=0.0022 | IA:p=0.247(NS)
    TOX          CD8_T    IT:+86.6% p=0.0022 | IA:p=0.126(NS)
    RB1          B        IT:+99.9% p=0.0043 | IA:p=0.429(NS)
    BCL6         CD8_T    IT:+163.1% p=0.0043 | IA:p=0.177(NS)
    TOX          Myeloid  IT:+1856.8% p=0.0043 | IA:p=0.908(NS)
    TP53         PlasmaB  IT:+380.2% p=0.0080 | IA:p=0.117(NS)
    EIF4EBP1     NK       IT:-53.1% p=0.0087 | IA:p=0.247(NS)
    TOX          CD4_T    IT:+100.7% p=0.0087 | IA:p=0.126(NS)
    LAYN         CD4_T    IT:+430.5% p=0.0087 | IA:p=0.126(NS)

  Blood: 85 IT-specific genes
    ATM          CD8_T    IT:+80.6% p=0.0025 | IA:p=0.286(NS)
    ATM          NK       IT:+78.4% p=0.0025 | IA:p=0.286(NS)
    ATM          CD4_T    IT:+60.2% p=0.0025 | IA:p=0.286(NS)
    CTLA4        CD8_T    IT

In [60]:
# ============================================================
#  SAVE C8 RESULTS TO CSV
# ============================================================
print("=" * 70)
print("  💾 SAVING C8 CHARACTERISTICS TO FILES")
print("=" * 70)

# Containers for aggregated results
c8_it_specific = []
c8_ia_transition = []
c8_cr_scar = []
c8_ia_ar = []

# Re-extract and accumulate data from both tissues
for tissue_name, df in [('Liver', df_gene_liver), ('Blood', df_gene_blood)]:
    # 1. IT-Specific (NL->IT sig, NL->IA NS)
    nl_it = df[df['comparison']=='NL→IT'].set_index(['gene','lineage'])
    nl_ia = df[df['comparison']=='NL→IA'].set_index(['gene','lineage'])
    common_it = nl_it.index.intersection(nl_ia.index)

    for idx in common_it:
        if nl_it.loc[idx, 'p_value'] < 0.05 and nl_ia.loc[idx, 'p_value'] > 0.10:
            c8_it_specific.append({
                'tissue': tissue_name, 'lineage': idx[1], 'gene': idx[0],
                'pathway': nl_it.loc[idx, 'pathway'],
                'IT_pct': nl_it.loc[idx, 'pct_change'], 'IT_p': nl_it.loc[idx, 'p_value'],
                'IA_p': nl_ia.loc[idx, 'p_value']
            })

    # 2. IA Transition (IT->IA sig)
    it_ia = df[(df['comparison']=='IT→IA') & (df['p_value']<0.05)]
    for _, row in it_ia.iterrows():
        c8_ia_transition.append(row.to_dict())

    # 3. CR Scar (NL->CR sig, NL->IA NS)
    nl_cr = df[df['comparison']=='NL→CR'].set_index(['gene','lineage'])
    common_cr = nl_cr.index.intersection(nl_ia.index)

    for idx in common_cr:
        if nl_cr.loc[idx, 'p_value'] < 0.05 and nl_ia.loc[idx, 'p_value'] > 0.10:
            c8_cr_scar.append({
                'tissue': tissue_name, 'lineage': idx[1], 'gene': idx[0],
                'pathway': nl_cr.loc[idx, 'pathway'],
                'CR_pct': nl_cr.loc[idx, 'pct_change'], 'CR_p': nl_cr.loc[idx, 'p_value'],
                'IA_p': nl_ia.loc[idx, 'p_value']
            })

    # 4. IA vs AR Discriminators (IA->AR sig)
    ia_ar = df[(df['comparison']=='IA→AR') & (df['p_value']<0.05)]
    for _, row in ia_ar.iterrows():
        c8_ia_ar.append(row.to_dict())

# Convert to DataFrames and Save
df_c8_it = pd.DataFrame(c8_it_specific)
df_c8_ia_trans = pd.DataFrame(c8_ia_transition)
df_c8_cr = pd.DataFrame(c8_cr_scar)
df_c8_ar = pd.DataFrame(c8_ia_ar)

df_c8_it.to_csv(f'{C8_SAVE}C8_IT_specific_genes.csv', index=False)
df_c8_ia_trans.to_csv(f'{C8_SAVE}C8_IA_transition_genes.csv', index=False)
df_c8_cr.to_csv(f'{C8_SAVE}C8_CR_scar_genes.csv', index=False)
df_c8_ar.to_csv(f'{C8_SAVE}C8_IA_AR_discriminators.csv', index=False)

# Verify
print(f"✅ Saved files to {C8_SAVE}:")
for f in sorted(os.listdir(C8_SAVE)):
    print(f"  - {f}")

print(f"\nSummary of saved features:")
print(f"  IT-Specific: {len(df_c8_it)} rows")
print(f"  IA-Transition: {len(df_c8_ia_trans)} rows")
print(f"  CR-Scar: {len(df_c8_cr)} rows")
print(f"  IA-AR Discriminators: {len(df_c8_ar)} rows")

  💾 SAVING C8 CHARACTERISTICS TO FILES
✅ Saved files to /content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics/:
  - C8_CR_scar_genes.csv
  - C8_IA_AR_discriminators.csv
  - C8_IA_transition_genes.csv
  - C8_IT_specific_genes.csv

Summary of saved features:
  IT-Specific: 122 rows
  IA-Transition: 54 rows
  CR-Scar: 114 rows
  IA-AR Discriminators: 48 rows


In [59]:
# %%
# ============================================================
#  C6-C8 COMPLETE
# ============================================================
print("=" * 70)
print("  ✅ C6-C8 COMPLETE")
print("=" * 70)

print(f"""
  C6 Files: {C6_SAVE}
  C7 Files: {C7_SAVE}
  C8 Files: {C8_SAVE}

  ▶ NEXT: C9 — Manuscript Preparation
  ▶ Order: Results → Discussion → Figures/Tables → M&M →
           Introduction → References → Title/Abstract/Key Points
""")


  ✅ C6-C8 COMPLETE

  C6 Files: /content/drive/MyDrive/ITLAS/results/version18-analysis/C6_discrepancy/
  C7 Files: /content/drive/MyDrive/ITLAS/results/version18-analysis/C7_patterns/
  C8 Files: /content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics/

  ▶ NEXT: C9 — Manuscript Preparation
  ▶ Order: Results → Discussion → Figures/Tables → M&M → 
           Introduction → References → Title/Abstract/Key Points

